# Proyecto Final — Derivados · Apple (AAPL): Bono Convertible Sintético con volatilidad GARCH
**Base metodológica:** caso *Molson Coors Brewing Co.: Valuing a Convertible Bond* (NE90-06-107) + clase de GARCH (Hull, 11.ª ed., caps. 13, 22 y 23)
**Formato de entrega:** dashboard tipo FINA 405 (tarjetas KPI + pestañas), HTML autocontenido
**Datos:** Refinitiv / LSEG (sesión abierta con tu App Key)

**Lógica.** Apple no tiene convertibles: tomamos un bono real de Apple como **bono puro** (su YTM es la tasa de descuento,
el análogo del 5.75% de Molson) y preguntamos qué cupón pagaría Apple si emitiera hoy ese bono como **convertible a 6 años**.
La versión 2 agrega el módulo de la clase: **GARCH(1,1) decide qué volatilidad usar** en la opción de conversión y el
**árbol CRR** mide cuánto vale poder convertir antes (opción americana).

| Paso | Caso Molson / Clase GARCH | Proyecto AAPL |
|---|---|---|
| 1 | Hull 23.1–23.5: histórica, EWMA, GARCH por MLE | Estimación con 5 años de AAPL (Refinitiv), validación con `arch` |
| 2 | Hull 23.4: elegir modelo (LL, AIC/BIC, Ljung-Box) | Tabla comparativa |
| 3 | Hull 23.6: volatilidad promedio para la vida de la opción | σ GARCH promedio a 6 años → entra al convertible |
| 4 | Hull 23.7: correlaciones | AAPL vs S&P 500 con EWMA |
| 5 | Hull 22.6 y 22.8: Monte Carlo GARCH, VaR y Kupiec | Abanico de precios, VaR 99% y back testing |
| 6 | Exhibit 2 del caso | Opción de conversión: BS, CRR americano y MC-GARCH (Duan) |
| 7 | Exhibit 3 | Cupón que deja el convertible a la par |
| 8 | Sensibilidades del caso | Prima × volatilidad |
| 9 | Exhibits 1 y 4 | Call spread: note hedge 25% + warrant 60% |
| 10 | Señal de volatilidad de la clase | σ GARCH vs σ implícita y straddle ATM |

> **Correr:** *Kernel → Restart & Run All*. Si un campo de Refinitiv no responde, el dato se marca **⚠ respaldo** en el dashboard.

In [ ]:
# ---- Silenciar NotOpenSSLWarning (urllib3 v2 + LibreSSL en macOS) — debe ir primero ----
import os, warnings
warnings.filterwarnings("ignore", message=r".*urllib3 v2 only supports OpenSSL.*")
warnings.filterwarnings("ignore", category=UserWarning, module=r"urllib3")
os.environ["PYTHONWARNINGS"] = "ignore:urllib3 v2 only supports OpenSSL"

In [ ]:
# ---- Instalación de paquetes (solo si faltan) ----
import subprocess, sys, importlib
for _mod, _pkg in [("plotly", "plotly"), ("scipy", "scipy"), ("arch", "arch"), ("statsmodels", "statsmodels"),
                   ("openpyxl", "openpyxl"), ("nbconvert", "nbconvert")]:
    try:
        importlib.import_module(_mod)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])

In [ ]:
import numpy as np, pandas as pd, datetime as dt, json
from scipy.stats import norm, chi2
from scipy.optimize import brentq, minimize
from scipy.signal import lfilter
import plotly.graph_objects as go
from plotly.subplots import make_subplots
warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1 · Parámetros del proyecto

In [ ]:
AUTORES   = "Equipo 3"
CURSO     = "Productos Derivados y Administración de Riesgos Corporativos · EGADE"
HOY       = dt.date.today()
TICKER_RIC = "AAPL.O"
BENCH_RIC  = ".SPX"                      # para la correlación (Hull 23.7)

BONO = dict(nombre="Apple Inc 4.500% Notes due 12-May-2032", cusip="037833FA3", isin="US037833FA32",
            cupon=0.045, vencimiento="2032-05-12", emision="2025-05-12", monto_usd=1_000_000_000,
            frecuencia=2, rango="Senior Unsecured", call="Make-whole; par call desde 12-Mar-2032")

UNIVERSO_RESPALDO = [
    dict(nombre="AAPL 4.000% 10-May-2028", cusip="037833ET3", isin="US037833ET32", cupon=0.0400, vencimiento="2028-05-10"),
    dict(nombre="AAPL 4.150% 10-May-2030", cusip="037833EU0", isin="US037833EU05", cupon=0.0415, vencimiento="2030-05-10"),
    dict(nombre="AAPL 4.500% 12-May-2032", cusip="037833FA3", isin="US037833FA32", cupon=0.0450, vencimiento="2032-05-12"),
    dict(nombre="AAPL 3.350% 08-Aug-2032", cusip="037833EP1", isin="US037833EP10", cupon=0.0335, vencimiento="2032-08-08"),
]

# Convertible (caso)
PLAZO_CONV      = 6.0
PRIMA_CONV      = 0.25
PRIMA_WARRANT   = 0.60
AJUSTE_VOL_WARR = -0.02
PRIMAS_SENS     = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60]
USAR_DIVIDENDO  = True
MONTO_EMISION   = None

# Volatilidad: cuál entra al convertible → "GARCH" (promedio a 6 años), "HIST" (1 año), "EWMA" o "IV"
VOL_FUENTE      = "GARCH"

# Módulo GARCH (clase)
YEARS        = 5
LAMBDA_EWMA  = 0.94
HORIZON_DAYS = 126            # pronóstico de precio y opción ATM (~6 meses)
N_SIM        = 5000
MU_ANUAL     = 0.08           # rendimiento esperado SUPUESTO (solo para el pronóstico del precio)
SEED         = 123
UMBRAL_SENAL = 0.10
IV_MANUAL    = None           # p. ej. 0.265 si quieres fijar la implícita ATM a mano
N_ARBOL      = 500

# Respaldo — SOLO si Refinitiv no devuelve el dato (se marca ⚠)
RESPALDO = dict(S=347.10, precio_bono=96.894, div_yield=0.0032, iv=0.265,
                ust={2:0.047, 3:0.048, 5:0.0492, 7:0.050, 10:0.0508, 30:0.053})
CARPETA = "."

## 2 · Conexión a Refinitiv y utilidades

In [ ]:
import lseg.data as ld
from datetime import datetime
from getpass import getpass

APP_KEY = getpass("App Key: ")
USERNAME = input("Usuario LSEG: ")
PASSWORD = getpass("Contraseña LSEG: ")

API = None
try:
    grant = ld.session.platform.GrantPassword(username=USERNAME, password=PASSWORD)
    definicion = ld.session.platform.Definition(app_key=APP_KEY, grant=grant, signon_control=True)
    session = definicion.get_session()
    session.open()
    ld.session.set_default(session)
    print("Estado:", session.open_state, "| Hora:", datetime.now())
    rd = ld
    prueba = rd.get_data("AAPL.O", ["TR.PriceClose"])
    if prueba is not None and not prueba.empty:
        API = "rd"
        print("CONECTADO: AAPL =", prueba.iloc[0, -1])
except Exception as e:
    print("ERROR DE CONEXION:", e)
PASSWORD = None
print("API:", API)

In [ ]:
FUENTES = {}

def registra(clave, valor, fuente):
    FUENTES[clave] = fuente
    return valor

def num(v):
    try:
        return float(v)
    except Exception:
        return np.nan

def en_pct(v):
    v = num(v)
    return v / 100 if pd.notna(v) else v

def _ok(v):
    if v is None:
        return False
    return str(v).strip() not in ("", "<NA>", "nan", "None")

def get_data(instr, fields):
    if API is None:
        return pd.DataFrame()
    if isinstance(instr, str):
        instr = [instr]
    if isinstance(fields, str):
        fields = [fields]
    try:
        df = rd.get_data(list(instr), list(fields))
        return pd.DataFrame() if df is None else df
    except Exception as e:
        print("  sin dato", fields, str(e)[:80])
        return pd.DataFrame()

def first_value(instr, fields):
    for f in fields:
        df = get_data(instr, [f])
        if df.shape[1] >= 2 and len(df) > 0:
            v = df.iloc[0, -1]
            if _ok(v):
                return v, f
    return None, None

def batch_value(instrs, fields):
    out = {i: np.nan for i in instrs}
    for f in fields:
        faltan = [i for i in instrs if pd.isna(out[i])]
        if len(faltan) == 0:
            break
        df = get_data(faltan, [f])
        if df.shape[1] < 2:
            continue
        for ins, v in zip(df.iloc[:, 0], df.iloc[:, -1]):
            if ins in out and _ok(v):
                out[ins] = num(v)
    return out

def get_history(ric, start, end=None):
    if API is None:
        return pd.Series(dtype=float)
    end = end or HOY.isoformat()
    for campos in (["TRDPRC_1"], ["TR.PriceClose"]):
        try:
            df = rd.get_history(universe=ric, fields=campos, interval="daily", start=start, end=end)
            df = df.apply(pd.to_numeric, errors="coerce").dropna(axis=1, how="all")
            if not df.empty:
                s = df.iloc[:, 0].dropna()
                s.index = pd.to_datetime(s.index)
                return s
        except Exception as e:
            print("  historia", ric, str(e)[:80])
    return pd.Series(dtype=float)

print("Utilidades listas. API =", API)

## 3 · Motor de valuación: Black-Scholes-Merton, bonos, árbol CRR y GARCH

In [ ]:
# ---------- Black-Scholes-Merton
def bsm(S, K, r, sig, T, q=0.0):
    S = np.asarray(S, float); sq = sig*np.sqrt(T)
    d1 = (np.log(S/K) + (r - q + 0.5*sig**2)*T)/sq; d2 = d1 - sq
    dq, dr = np.exp(-q*T), np.exp(-r*T)
    c = S*dq*norm.cdf(d1) - K*dr*norm.cdf(d2); p = c - S*dq + K*dr
    return dict(call=c, put=p, d1=d1, d2=d2, delta_c=dq*norm.cdf(d1), delta_p=dq*(norm.cdf(d1)-1),
                gamma=dq*norm.pdf(d1)/(S*sq), vega=S*dq*norm.pdf(d1)*np.sqrt(T)/100,
                theta_c=(-S*dq*norm.pdf(d1)*sig/(2*np.sqrt(T)) - r*K*dr*norm.cdf(d2) + q*S*dq*norm.cdf(d1))/365,
                rho_c=K*T*dr*norm.cdf(d2)/100)

# ---------- Bonos
def flujos(cupon, T, face=1000, f=2):
    n = int(np.ceil(T*f - 1e-9)); t = np.sort(T - np.arange(n)/f); t = t[t > 1e-9]
    cf = np.full(len(t), face*cupon/f); cf[-1] += face
    return t, cf

def valor_bono(cupon, y, T, face=1000, f=2, limpio=True):
    t, cf = flujos(cupon, T, face, f); sucio = np.sum(cf/(1+y/f)**(f*t))
    if not limpio: return sucio
    return sucio - (face*cupon/f*(1 - t[0]*f) if t[0] < 1/f else 0.0)

def ytm_de_precio(precio_100, cupon, T, f=2):
    return brentq(lambda y: valor_bono(cupon, y, T, 100, f) - precio_100, -0.05, 0.5)

def dur_conv(cupon, y, T, f=2):
    t, cf = flujos(cupon, T, 100, f); pv = cf/(1+y/f)**(f*t); P = pv.sum()
    mac = np.sum(t*pv)/P
    return mac, mac/(1+y/f), np.sum(t*(t+1/f)*pv)/(P*(1+y/f)**2)

def cupon_para_valor(objetivo, y, T, face=1000, f=2):
    v0 = valor_bono(0.0, y, T, face, f, False); v1 = valor_bono(0.01, y, T, face, f, False)
    return (objetivo - v0)/(v1 - v0)*0.01

def valuar_convertible(S, prima, r, sig, T, y, q=0.0, face=1000):
    K = S*(1+prima); n = face/K; o = bsm(S, K, r, sig, T, q)
    opc = o["call"]*n; piso = face - opc
    return dict(K=K, n=n, call_acc=float(o["call"]), put_acc=float(o["put"]), delta=float(o["delta_c"]),
                opcion_bono=float(opc), piso=float(piso), cupon=cupon_para_valor(piso, y, T, face), greeks=o)

# ---------- Árbol CRR (Hull cap. 13) con griegas
def crr(S0, K, r, sigma, T, N=500, tipo="call", americana=True, q=0.0):
    dt_ = T/N; u_ = np.exp(sigma*np.sqrt(dt_)); d_ = 1/u_
    p = (np.exp((r-q)*dt_) - d_)/(u_ - d_); disc = np.exp(-r*dt_); sg = 1 if tipo == "call" else -1
    niveles = [None]*3; j = np.arange(N+1)
    V = np.maximum(sg*(S0*u_**(N-j)*d_**j - K), 0.0)
    for i in range(N-1, -1, -1):
        k = np.arange(i+1); cont = disc*(p*V[:i+1] + (1-p)*V[1:i+2])
        V = np.maximum(cont, np.maximum(sg*(S0*u_**(i-k)*d_**k - K), 0.0)) if americana else cont
        if i <= 2: niveles[i] = V
    return V[0], niveles, u_, d_, dt_

def griegas_crr(S0, K, r, sigma, T, N=500, tipo="call", americana=True, q=0.0):
    precio, niv, u_, d_, dt_ = crr(S0, K, r, sigma, T, N, tipo, americana, q)
    f0, f1, f2 = niv[0][0], niv[1], niv[2]
    delta = (f1[0]-f1[1])/(S0*u_ - S0*d_)
    gamma = ((f2[0]-f2[1])/(S0*u_**2 - S0) - (f2[1]-f2[2])/(S0 - S0*d_**2))/(0.5*(S0*u_**2 - S0*d_**2))
    return dict(precio=precio, delta=delta, gamma=gamma, theta=(f2[1]-f0)/(2*dt_)/365,
                vega=crr(S0, K, r, sigma+0.01, T, N, tipo, americana, q)[0] - precio,
                rho=crr(S0, K, r+0.01, sigma, T, N, tipo, americana, q)[0] - precio)

# ---------- GARCH(1,1), EWMA y verosimilitud (Hull 23.2–23.5); rendimientos en %
def var_garch(params, u, v0=None):
    omega, alpha, beta = params
    v0 = np.var(u) if v0 is None else v0
    y, _ = lfilter([1.0], [1.0, -beta], omega + alpha*u[:-1]**2, zi=[beta*v0])
    return np.r_[v0, y]

def var_ewma(u, lam=LAMBDA_EWMA): return var_garch((0.0, 1-lam, lam), u, v0=np.var(u[:30]))
def loglik(v, u): return -0.5*np.sum(np.log(2*np.pi) + np.log(v) + u**2/v)
def vol_anual(v): return np.sqrt(252*np.asarray(v))/100

def ajustar_garch(u):
    v = np.var(u)
    res = minimize(lambda p: -loglik(var_garch(p, u), u), [v*0.03, 0.08, 0.90], method="SLSQP",
                   bounds=[(1e-6, None), (1e-4, 0.5), (0.30, 0.9998)],
                   constraints=({"type": "ineq", "fun": lambda p: 0.9999 - p[1] - p[2]},),
                   options={"maxiter": 500, "ftol": 1e-10})
    return res.x

def kupiec(x, N, p):
    if x == 0: lr = -2*N*np.log(1-p)
    else:
        ph = x/N; lr = -2*((N-x)*np.log((1-p)/(1-ph)) + x*np.log(p/ph))
    return float(1 - chi2.cdf(lr, 1))

## 4 · Validación: replicar el caso Molson Coors (Exhibits 2, 3 y 4)

In [ ]:
m = valuar_convertible(87.61, 0.25, 0.0446, 0.23, 6, 0.0575, q=0.0)
w = bsm(87.61, 87.61*1.60, 0.0446, 0.21, 6)
VALID = pd.DataFrame([
    ["Precio de conversión", 109.51, m["K"]], ["Acciones por bono", 9.13, m["n"]],
    ["Call por acción (25%)", 21.0, m["call_acc"]], ["Delta call", 0.64, m["delta"]],
    ["Put por paridad", 17.2, m["put_acc"]], ["Opción de conversión/bono", 191.5, m["opcion_bono"]],
    ["Valor bono puro", 808.5, m["piso"]], ["Cupón a la par (%)", 1.93, m["cupon"]*100],
    ["Warrant 60% por acción", 11.4, float(w["call"])], ["Costo note hedge (USD M)", 110.0, m["call_acc"]*5.25],
    ["Ingreso warrants (USD M)", 59.0, float(w["call"])*5.25],
], columns=["Concepto", "Caso", "Modelo"])
VALID["Diferencia"] = VALID["Modelo"] - VALID["Caso"]
VALID

## 5 · Universo de bonos USD de Apple en Refinitiv y selección

In [ ]:
def buscar_bonos_aapl():
    if API != "rd": return pd.DataFrame()
    sel = "RIC,ISIN,CUSIP,IssuerLegalName,CouponRate,MaturityDate,IssueDate,FaceIssuedTotal,RCSCurrencyLeaf"
    for flt in ["IsActive eq true and RCSCurrencyLeaf eq 'US Dollar'", "IsActive eq true", None]:
        try:
            kw = dict(view=rd.discovery.Views.GOV_CORP_INSTRUMENTS, query="Apple Inc", select=sel, top=300)
            if flt: kw["filter"] = flt
            df = rd.discovery.search(**kw)
            if df is not None and not df.empty:
                df = df[df.astype(str).agg(" ".join, axis=1).str.contains("Apple Inc", case=False)]
                if "RCSCurrencyLeaf" in df: df = df[df["RCSCurrencyLeaf"].astype(str).str.contains("US Dollar")]
                if not df.empty: return df
        except Exception as e:
            print("  · discovery.search:", str(e)[:100])
    return pd.DataFrame()

raw = buscar_bonos_aapl()
if not raw.empty:
    U = pd.DataFrame({
        "ric": raw.get("RIC"), "isin": raw.get("ISIN"), "cusip": raw.get("CUSIP"),
        "cupon": pd.to_numeric(raw.get("CouponRate"), errors="coerce")/100,
        "vencimiento": pd.to_datetime(raw.get("MaturityDate"), errors="coerce").dt.date,
        "monto": pd.to_numeric(raw.get("FaceIssuedTotal"), errors="coerce")})
    FUENTE_UNIV = "Refinitiv discovery.search (GOV_CORP_INSTRUMENTS)"
else:
    U = pd.DataFrame(UNIVERSO_RESPALDO).drop(columns="nombre")
    U["vencimiento"] = pd.to_datetime(U["vencimiento"]).dt.date; U["ric"] = None; U["monto"] = np.nan
    FUENTE_UNIV = "⚠ respaldo (term sheets SEC)"

U = U.dropna(subset=["cupon", "vencimiento"])

U["nombre"] = [f"AAPL {c*100:.3f}% {v:%d-%b-%Y}" for c, v in zip(U["cupon"], U["vencimiento"])]
U["anios"]  = (pd.to_datetime(U["vencimiento"]) - pd.Timestamp(HOY)).dt.days/365.25
U = U[pd.to_datetime(U["vencimiento"]) > pd.Timestamp(HOY)].drop_duplicates("isin")
U = U[(pd.to_datetime(U["vencimiento"]) - pd.Timestamp(HOY)).dt.days.between(365, 365*12)].sort_values("vencimiento").reset_index(drop=True)
# Precios: en lote con TR.MidPrice (el campo que sí respondió) y respaldo por CUSIP= / ISIN
U["id_px"] = U["cusip"].astype(str) + "="
px = batch_value(U["id_px"].tolist(), ["TR.MidPrice", "MID_PRICE", "CF_CLOSE", "TR.BIDPRICE"])
U["precio"] = U["id_px"].map(px)
falt = U["precio"].isna() & U["isin"].notna()
if falt.any() and API:
    px2 = batch_value(U.loc[falt, "isin"].tolist(), ["TR.MidPrice", "TR.BIDPRICE"])
    U.loc[falt, "precio"] = U.loc[falt, "isin"].map(px2)
U["ytm"] = [ytm_de_precio(p, c, T) if pd.notna(p) and p > 0 else np.nan for p, c, T in zip(U["precio"], U["cupon"], U["anios"])]

U["dist_plazo"] = (U["anios"] - PLAZO_CONV).abs()
U["dist_par"]   = (U["precio"].fillna(100) - 100).abs()
U["puntaje"]    = (100 - 12*U["dist_plazo"] - 2*U["dist_par"]).round(1)
print("Fuente universo:", FUENTE_UNIV, "| bonos con precio:", int(U["precio"].notna().sum()), "de", len(U))
U.sort_values("puntaje", ascending=False)[["nombre", "cusip", "isin", "cupon", "anios", "precio", "ytm", "puntaje"]]

## 6 · Datos de mercado: bono elegido, Treasuries, acción, S&P 500 y dividendo

In [ ]:
T_BONO = (pd.Timestamp(BONO["vencimiento"]) - pd.Timestamp(HOY)).days/365.25
precio_bono = fuente_p = None
for i in [BONO["cusip"] + "=", BONO["isin"]]:
    precio_bono, fuente_p = first_value(i, ["TR.MidPrice", "MID_PRICE", "CF_CLOSE", "CF_LAST"])
    if precio_bono is not None: break
precio_bono = registra("precio_bono", num(precio_bono), f"Refinitiv {fuente_p}") if precio_bono is not None \
              else registra("precio_bono", RESPALDO["precio_bono"], "⚠ respaldo")
YTM = registra("ytm", ytm_de_precio(precio_bono, BONO["cupon"], T_BONO), f"Calculado de precio ({FUENTES['precio_bono']})")
MAC, MOD, CVX = dur_conv(BONO["cupon"], YTM, T_BONO)

UST = {}
for plazo in [2, 3, 5, 7, 10, 30]:
    v, fld = first_value(f"US{plazo}YT=RR", ["MID_YLD_1", "B_YLD_1", "RT_YIELD_1", "TR.MidYield"])
    UST[plazo] = en_pct(v) if v is not None else RESPALDO["ust"][plazo]
    FUENTES[f"ust{plazo}"] = f"Refinitiv {fld}" if v is not None else "⚠ respaldo"
ust_x, ust_y = zip(*sorted(UST.items()))
rf = lambda T: float(np.interp(T, ust_x, ust_y))
RF = registra("rf", rf(PLAZO_CONV), f"UST interpolado a {PLAZO_CONV:.0f}a ({FUENTES['ust5']})")
R_6M = rf(0.5) if min(ust_x) <= 0.5 else UST[2]          # tasa para la opción ATM de ~6 meses
SPREAD = YTM - rf(T_BONO)

S, fs = first_value(TICKER_RIC, ["CF_LAST", "TRDPRC_1", "TR.PriceClose"])
S = registra("S", num(S), f"Refinitiv {fs}") if S is not None else registra("S", RESPALDO["S"], "⚠ respaldo")

# Dividend yield: Refinitiv lo entrega en % → ÷100 (corrige el error de la versión 1)
dy, fdy = first_value(TICKER_RIC, ["TR.DividendYield", "YIELD"])
Q = registra("q", en_pct(dy), f"Refinitiv {fdy} (÷100)") if dy is not None else registra("q", RESPALDO["div_yield"], "⚠ respaldo")
if not (0 <= Q < 0.05):
    print(f"⚠ dividend yield fuera de rango ({Q}); uso respaldo"); Q = registra("q", RESPALDO["div_yield"], "⚠ respaldo")
Q_USO = Q if USAR_DIVIDENDO else 0.0
MONTO = MONTO_EMISION or BONO["monto_usd"]

# Historia: AAPL y S&P 500 (5 años)
inicio = (HOY - dt.timedelta(days=int(YEARS*365.25) + 10)).isoformat()
h_aapl, h_spx = get_history(TICKER_RIC, inicio), get_history(BENCH_RIC, inicio)

def simular_historia(n=1255):
    """Respaldo didáctico: GARCH simulado (solo si no hay Refinitiv)."""
    rng = np.random.default_rng(SEED); z = rng.standard_normal((n, 2)); z[:, 1] = 0.6*z[:, 0] + 0.8*z[:, 1]
    cols = {}
    for j, (nom, (om, al, be), s_fin) in enumerate([("AAPL", (0.10, 0.062, 0.905), S), ("SPX", (0.03, 0.08, 0.90), 6500.0)]):
        v = om/(1-al-be); u = np.empty(n)
        for i in range(n): u[i] = np.sqrt(v)*z[i, j]; v = om + al*u[i]**2 + be*v
        p = np.exp(np.cumsum(u/100 + 0.0003)); cols[nom] = p/p[-1]*s_fin
    return pd.DataFrame(cols, index=pd.bdate_range(end=pd.Timestamp(HOY), periods=n))

if h_aapl.empty:
    PRECIOS = simular_historia(); FUENTE_HIST = "⚠ DATOS SIMULADOS (sin Refinitiv)"
else:
    PRECIOS = pd.concat({"AAPL": h_aapl, "SPX": h_spx}, axis=1).dropna(how="all")
    if PRECIOS["SPX"].isna().all(): PRECIOS = PRECIOS.drop(columns="SPX")
    PRECIOS = PRECIOS.dropna(); FUENTE_HIST = "Refinitiv get_history"
FUENTES["historia"] = FUENTE_HIST
HIST = PRECIOS["AAPL"]
REND = 100*np.log(PRECIOS/PRECIOS.shift(1)).dropna()          # rendimientos en %
print(f"Historia: {FUENTE_HIST} · {len(PRECIOS)} obs · {PRECIOS.index[0].date()} → {PRECIOS.index[-1].date()}")

pd.Series({"Precio limpio bono": precio_bono, "YTM": YTM, "Duración modificada": MOD, "rf 6a": RF, "Spread vs UST": SPREAD,
           "AAPL spot": S, "Dividend yield": Q})

## 7 · Empresa (KPIs FINA 405) y contexto China

In [ ]:
CAMPOS_EMP = {
    "Market Cap": ["TR.CompanyMarketCap"], "Revenue (LTM)": ["TR.Revenue"], "EBITDA": ["TR.EBITDA"],
    "Net Income": ["TR.NetIncome", "TR.NetIncomeBeforeExtraItems"], "Total Debt": ["TR.TotalDebtOutstanding", "TR.TotalDebt"],
    "Total Assets": ["TR.TotalAssetsReported", "TR.TotalAssets"], "Total Equity": ["TR.TotalEquity", "TR.ShareholdersEquity"],
    "Cash & ST Inv.": ["TR.CashAndSTInvestments", "TR.CashandEquivalents"],
    "Interest Expense": ["TR.InterestExpense", "TR.F.IntrExpn", "TR.F.IntExpnTot", "TR.InterestExpNonOperating"],
    "EBIT": ["TR.EBIT"], "Shares Out.": ["TR.SharesOutstanding"],
}
EMP = {}
for k, flds in CAMPOS_EMP.items():
    v, f = first_value(TICKER_RIC, flds)
    EMP[k] = num(v) if v is not None else np.nan
    FUENTES[f"emp_{k}"] = f"Refinitiv {f}" if f else "n/d"

def ratio(a, b):
    return EMP[a]/EMP[b] if pd.notna(EMP.get(a)) and pd.notna(EMP.get(b)) and EMP[b] else np.nan
EMP["D/E"] = ratio("Total Debt", "Total Equity"); EMP["Debt/Assets"] = ratio("Total Debt", "Total Assets")
# Apple no reporta gasto por intereses por separado; si falta, se estima como deuda × YTM y se rotula "estimada"
COB_EST = pd.isna(EMP["Interest Expense"])
int_exp = EMP["Total Debt"]*YTM if COB_EST else abs(EMP["Interest Expense"])
EMP["Interest Coverage"] = EMP["EBIT"]/int_exp if pd.notna(int_exp) and int_exp else np.nan
EMP["Net Debt/EBITDA"] = (EMP["Total Debt"] - EMP["Cash & ST Inv."])/EMP["EBITDA"] if pd.notna(EMP["EBITDA"]) else np.nan
EMP["EBIT Margin"] = ratio("EBIT", "Revenue (LTM)"); EMP["Net Margin"] = ratio("Net Income", "Revenue (LTM)")
EMP["Dilución conv. 25%"] = (MONTO/(S*(1+PRIMA_CONV)))/EMP["Shares Out."] if pd.notna(EMP["Shares Out."]) else np.nan

PAIS = {}
v, _ = first_value("CNY=", ["CF_LAST", "MID_PRICE", "TR.MidPrice"]);         PAIS["USD/CNY"] = num(v) if v is not None else np.nan
v, _ = first_value("CN10YT=RR", ["MID_YLD_1", "B_YLD_1", "TR.MidYield"]);   PAIS["China 10Y"] = en_pct(v) if v is not None else np.nan
PAIS["US 10Y"] = UST[10]
pd.Series({**EMP, **PAIS})

## 8 · Volatilidad: histórica, EWMA y GARCH(1,1) por máxima verosimilitud (Hull 23.1–23.5)
$$\sigma_n^2=\omega+\alpha u_{n-1}^2+\beta\sigma_{n-1}^2,\qquad V_L=\frac{\omega}{1-\alpha-\beta}$$
α = peso del shock de ayer · β = memoria · α+β = persistencia · $V_L$ = varianza de largo plazo.

In [ ]:
u = REND["AAPL"].values
VOL_30 = REND["AAPL"].rolling(30).std()*np.sqrt(252)/100
VOL_HIST_1A = float(REND["AAPL"].iloc[-252:].std()*np.sqrt(252)/100)

P_G = ajustar_garch(u); OMEGA, ALPHA, BETA = P_G
V_G = var_garch(P_G, u); VL = OMEGA/(1-ALPHA-BETA)
V_NEXT = OMEGA + ALPHA*u[-1]**2 + BETA*V_G[-1]
V_EW = var_ewma(u); VOL_EWMA_HOY = float(vol_anual(LAMBDA_EWMA*V_EW[-1] + (1-LAMBDA_EWMA)*u[-1]**2))
VIDA_MEDIA = np.log(0.5)/np.log(ALPHA+BETA)

GARCH_TAB = pd.Series({"omega": OMEGA, "alpha": ALPHA, "beta": BETA, "alpha+beta": ALPHA+BETA,
                       "Vol largo plazo (anual)": float(vol_anual(VL)), "Vol hoy GARCH (anual)": float(vol_anual(V_NEXT)),
                       "Vol EWMA hoy (anual)": VOL_EWMA_HOY, "Vol histórica 1 año": VOL_HIST_1A, "Vida media del shock (días)": VIDA_MEDIA})
try:
    from arch import arch_model
    am = arch_model(u, mean="Zero", vol="GARCH", p=1, q=1, dist="normal").fit(disp="off")
    ARCH_CHECK = f"Paquete arch: ω={am.params['omega']:.4f} α={am.params['alpha[1]']:.4f} β={am.params['beta[1]']:.4f} | MLE propio: {OMEGA:.4f} {ALPHA:.4f} {BETA:.4f}"
except Exception as e:
    ARCH_CHECK = f"Validación con arch omitida: {e}"
print(ARCH_CHECK)
print(f"Persistencia {ALPHA+BETA:.3f}: un shock pierde la mitad de su efecto en ~{VIDA_MEDIA:.0f} días. "
      f"Vol largo plazo {vol_anual(VL):.1%}; hoy {vol_anual(V_NEXT):.1%}.")
GARCH_TAB

## 9 · ¿Qué modelo elegir? (Hull 23.4) — LL más alta, AIC/BIC más bajos y Ljung-Box p > 0.05

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox
def lb_p(serie, lags=10): return float(acorr_ljungbox(serie, lags=[lags], return_df=True)["lb_pvalue"].iloc[0])

n_obs = len(u)
grid_l = np.arange(0.80, 0.995, 0.005)
LAM_OPT = grid_l[int(np.argmax([loglik(var_ewma(u, l), u) for l in grid_l]))]
modelos = {"Volatilidad constante": (np.full(n_obs, np.var(u)), 1), "EWMA λ=0.94": (V_EW, 0),
           f"EWMA λ óptimo ({LAM_OPT:.3f})": (var_ewma(u, LAM_OPT), 1), "GARCH(1,1)": (V_G, 3)}
COMPAR = pd.DataFrame([{"Modelo": k, "Log-verosim.": loglik(v, u), "AIC": 2*kk - 2*loglik(v, u),
                        "BIC": kk*np.log(n_obs) - 2*loglik(v, u), "p Ljung-Box (u²/σ²)": lb_p(u**2/v)}
                       for k, (v, kk) in modelos.items()])
MEJOR_MODELO = COMPAR.loc[COMPAR["AIC"].idxmin(), "Modelo"]
print("u² sin modelar, p Ljung-Box:", round(lb_p(u**2), 4), "(p pequeño = hay clustering de volatilidad)")
print("Mejor modelo por AIC:", MEJOR_MODELO)
COMPAR

## 10 · Pronóstico GARCH y volatilidad para el convertible (Hull 23.6)
$$E[\sigma^2_{n+k}]=V_L+(\alpha+\beta)^k(\sigma^2_{n}-V_L)$$
Para una opción de vida $T$ se usa la **volatilidad promedio** esperada en esos $T$ días. El convertible vive 6 años (1,512 días hábiles).

In [ ]:
def pron_var(H):
    k = np.arange(H); return VL + (ALPHA+BETA)**k*(V_NEXT - VL)
def vol_promedio(dias): return float(np.sqrt(252*pron_var(max(int(dias), 1)).mean())/100)
def vol_promedio_hull(dias):
    a = np.log(1/(ALPHA+BETA)); return float(np.sqrt(252*(VL + (1-np.exp(-a*dias))/(a*dias)*(V_NEXT-VL)))/100)

PLAZOS = [10, 30, 63, 126, 252, 756, 1512]
EST_TEMP = pd.DataFrame({"Vol promedio GARCH": [vol_promedio(d) for d in PLAZOS],
                         "Fórmula continua Hull": [vol_promedio_hull(d) for d in PLAZOS]},
                        index=[f"{d} días" for d in PLAZOS])
VOL_GARCH_6A = vol_promedio(PLAZO_CONV*252)
VOL_GARCH_6M = vol_promedio(HORIZON_DAYS)
EST_TEMP

## 11 · Correlación AAPL – S&P 500 con EWMA (Hull 23.7)

In [ ]:
if "SPX" in REND:
    x, y = REND["AAPL"].values, REND["SPX"].values
    cov = np.empty(len(x)); cov[0] = np.mean(x[:30]*y[:30])
    for i in range(1, len(x)): cov[i] = LAMBDA_EWMA*cov[i-1] + (1-LAMBDA_EWMA)*x[i-1]*y[i-1]
    CORR_EWMA = pd.Series(cov/np.sqrt(var_ewma(x)*var_ewma(y)), index=REND.index)
    CORR_60 = REND["AAPL"].rolling(60).corr(REND["SPX"])
    BETA_MKT = np.cov(x[-252:], y[-252:])[0, 1]/np.var(y[-252:])
    print(f"Correlación EWMA hoy {CORR_EWMA.iloc[-1]:.2f} · promedio {CORR_EWMA.mean():.2f} · beta 1 año {BETA_MKT:.2f}")
else:
    CORR_EWMA = CORR_60 = pd.Series(dtype=float); BETA_MKT = np.nan
    print("Sin historia del S&P 500: se omite la correlación")

## 12 · Monte Carlo con GARCH: abanico de precios y VaR (Hull 22.6)

In [ ]:
def simular(S0, H, n_sim, deriva_anual, rng):
    v = np.full(n_sim, V_NEXT); Sx = np.empty((H+1, n_sim)); Sx[0] = S0
    for k in range(1, H+1):
        sig = np.sqrt(v); z = rng.standard_normal(n_sim)
        Sx[k] = Sx[k-1]*np.exp(deriva_anual/252 - 0.5*(sig/100)**2 + (sig/100)*z)
        v = OMEGA + ALPHA*(sig*z)**2 + BETA*v
    return Sx

rng = np.random.default_rng(SEED)
SIM = simular(S, HORIZON_DAYS, N_SIM, MU_ANUAL, rng)
ST = SIM[-1]; r10 = SIM[10]/S - 1
MC = pd.Series({"Precio hoy": S, **dict(zip(["P5", "P25", "Mediana", "P75", "P95"], np.percentile(ST, [5, 25, 50, 75, 95]))),
                "Prob. sube": (ST > S).mean(), "VaR 99% 10 días (%)": -np.percentile(r10, 1),
                "VaR 99% 10 días USD (1 bono convertible)": -np.percentile(r10, 1)*1000})
MC

## 13 · Back testing del VaR con la prueba de Kupiec (Hull 22.8) — GARCH reestimado cada 63 días

In [ ]:
def backtest(nivel, inicio, refit=63):
    n = len(u); z = norm.ppf(nivel)
    pred = {"GARCH(1,1)": np.full(n, np.nan), "EWMA": V_EW, "Histórica 250d": np.full(n, np.nan)}
    for s in range(inicio, n, refit):
        fin = min(n, s+refit); pred["GARCH(1,1)"][s:fin] = var_garch(ajustar_garch(u[:s]), u[:fin], v0=np.var(u[:s]))[s:fin]
    for s in range(inicio, n): pred["Histórica 250d"][s] = np.var(u[s-250:s])
    filas = []
    for nom, vp in pred.items():
        exc = u[inicio:] < -z*np.sqrt(vp[inicio:]); N, xx = len(exc), int(exc.sum())
        filas.append({"Confianza": f"{nivel:.0%}", "Modelo": nom, "Días": N, "Excepciones": xx,
                      "Tasa observada": xx/N, "Tasa esperada": 1-nivel, "p Kupiec": kupiec(xx, N, 1-nivel),
                      "Resultado": "Pasa" if kupiec(xx, N, 1-nivel) > 0.05 else "No pasa"})
    return filas, pred["GARCH(1,1)"], z

INICIO_BT = int(len(u)*0.6); filas_bt = []
for nivel in (0.95, 0.99):
    f_, VG_BT, Z_BT = backtest(nivel, INICIO_BT); filas_bt += f_
BT = pd.DataFrame(filas_bt)
BT

## 14 · Opción de conversión: ¿qué volatilidad usar? (Exhibit 2 + clase GARCH)

In [ ]:
IV, fiv = (IV_MANUAL, "Manual") if IV_MANUAL else first_value(TICKER_RIC, ["TR.30DayATMIVCall", "TR.ImpliedVolatility", "TR.HV30"])
if IV is not None:
    IV = num(IV); IV = IV/100 if IV > 1.5 else IV
    IV = registra("iv", IV, f"Refinitiv {fiv}" if fiv != "Manual" else "Manual")
else:
    try:                                                     # mismo método de la clase (Yahoo Finance)
        import yfinance as yf
        tk = yf.Ticker("AAPL"); obj = pd.Timestamp.today() + pd.Timedelta(days=int(HORIZON_DAYS/252*365))
        exp = min(tk.options, key=lambda e: abs(pd.Timestamp(e) - obj)); ch = tk.option_chain(exp)
        c_ = ch.calls.iloc[(ch.calls.strike - S).abs().argsort().iloc[0]]; p_ = ch.puts.iloc[(ch.puts.strike - S).abs().argsort().iloc[0]]
        ivs = [x for x in (float(c_.impliedVolatility), float(p_.impliedVolatility)) if x > 0.05]
        IV = registra("iv", float(np.mean(ivs)), f"Yahoo Finance ATM {exp}")
    except Exception:
        IV = registra("iv", RESPALDO["iv"], "⚠ respaldo")

VOLS_CAND = {"HIST": ("Histórica 1 año", VOL_HIST_1A), "EWMA": ("EWMA hoy (λ=0.94)", VOL_EWMA_HOY),
             "GARCH": ("GARCH promedio 6 años", VOL_GARCH_6A), "IV": ("Implícita ATM", IV)}
SIGMA = registra("vol", VOLS_CAND[VOL_FUENTE][1], VOLS_CAND[VOL_FUENTE][0])

VOL_COMP = pd.DataFrame([{"Volatilidad": nom, "σ": s_, **{k: v for k, v in valuar_convertible(S, PRIMA_CONV, RF, s_, PLAZO_CONV, YTM, Q_USO).items()
                          if k in ("opcion_bono", "piso", "cupon")}} for nom, s_ in VOLS_CAND.values()])
VOL_COMP["En uso"] = VOL_COMP["Volatilidad"] == VOLS_CAND[VOL_FUENTE][0]
print(f"σ en uso para el convertible: {SIGMA:.2%} ({VOLS_CAND[VOL_FUENTE][0]})")
VOL_COMP

## 15 · Valuación del convertible sintético AAPL (Exhibits 2 y 3)

In [ ]:
C  = valuar_convertible(S, PRIMA_CONV, RF, SIGMA, PLAZO_CONV, YTM, q=Q_USO)
C0 = valuar_convertible(S, PRIMA_CONV, RF, SIGMA, PLAZO_CONV, YTM, q=0.0)
gk = C["greeks"]
EXH2 = pd.DataFrame({
    "Por acción": [S, C["K"], RF, SIGMA, PLAZO_CONV, Q_USO, C["call_acc"], C["delta"], C["put_acc"], float(gk["gamma"]), float(gk["vega"]), float(gk["theta_c"]), float(gk["rho_c"])],
    "Por bono":   [C["n"]*S, 1000, RF, SIGMA, PLAZO_CONV, Q_USO, C["opcion_bono"], C["delta"], C["put_acc"]*C["n"], float(gk["gamma"])/C["n"], float(gk["vega"])*C["n"], float(gk["theta_c"])*C["n"], float(gk["rho_c"])*C["n"]],
}, index=["S", "X", "rf", "σ", "T", "q", "Call", "Delta", "Put (paridad)", "Gamma", "Vega (1pt)", "Theta (día)", "Rho (1pt)"])

t_cf, cf = flujos(C["cupon"], PLAZO_CONV); pv = cf/(1+YTM/2)**(2*t_cf)
EXH3 = pd.DataFrame({"Año": t_cf, "Cupón": 1000*C["cupon"]/2, "Principal": np.where(t_cf == t_cf[-1], 1000, 0), "Flujo": cf, "VP": pv})
print(f"Conversión ${C['K']:.2f} · {C['n']:.4f} acc/bono · opción ${C['opcion_bono']:.2f} · bono puro ${C['piso']:.2f}")
print(f"Cupón a la par {C['cupon']:.3%} vs straight {YTM:.3%} → ahorro {(YTM-C['cupon'])*1e4:.0f} pb = USD {(YTM-C['cupon'])*MONTO/1e6:,.1f} M anuales")
EXH2

## 16 · Conversión como opción americana (árbol CRR) y comprobación con Monte Carlo GARCH (Duan)
El tenedor puede convertir antes del vencimiento: el árbol compara en cada nodo convertir ya vs esperar.
Con dividendo bajo (q ≪ r) la prima de ejercicio anticipado de una call es casi cero, así que Black-Scholes es válido.

In [ ]:
am_ = griegas_crr(S, C["K"], RF, SIGMA, PLAZO_CONV, N_ARBOL, "call", True, Q_USO)
eu_ = crr(S, C["K"], RF, SIGMA, PLAZO_CONV, N_ARBOL, "call", False, Q_USO)[0]

rng = np.random.default_rng(SEED + 1)
S_rn = simular(S, int(PLAZO_CONV*252), N_SIM*2, RF - Q_USO, rng)[-1]
MC_CALL = float(np.exp(-RF*PLAZO_CONV)*np.maximum(S_rn - C["K"], 0).mean())
MC_SE = float(np.exp(-RF*PLAZO_CONV)*np.maximum(S_rn - C["K"], 0).std()/np.sqrt(len(S_rn)))

AMER = pd.DataFrame({
    "Por acción": [C["call_acc"], eu_, am_["precio"], am_["precio"] - eu_, MC_CALL],
    "Por bono":   [C["opcion_bono"], eu_*C["n"], am_["precio"]*C["n"], (am_["precio"] - eu_)*C["n"], MC_CALL*C["n"]],
}, index=["Black-Scholes-Merton (europea)", "Árbol CRR europeo", "Árbol CRR americano",
          "Prima ejercicio anticipado", "Monte Carlo GARCH (Duan, europea)"])
GRIEGAS_CRR = pd.Series({k: am_[k] for k in ("delta", "gamma", "theta", "vega", "rho")})
print(f"MC-GARCH ±{1.96*MC_SE:.2f} (IC 95%) · diferencia MC vs BS {100*(MC_CALL/C['call_acc']-1):+.1f}%")
AMER

## 17 · Sensibilidades: prima de conversión y volatilidad

In [ ]:
SENS = pd.DataFrame([{"Prima": p, **{k: v for k, v in valuar_convertible(S, p, RF, SIGMA, PLAZO_CONV, YTM, Q_USO).items()
                                     if k in ("K", "opcion_bono", "piso", "cupon", "delta")}} for p in PRIMAS_SENS])
SENS["Ahorro (pb)"] = (YTM - SENS["cupon"])*1e4
SENS["Ahorro anual USD M"] = (YTM - SENS["cupon"])*MONTO/1e6
VOL_GRID = np.round(np.arange(max(0.10, SIGMA-0.10), SIGMA+0.105, 0.025), 3)
HEAT = pd.DataFrame([[valuar_convertible(S, p, RF, v, PLAZO_CONV, YTM, Q_USO)["cupon"]*100 for v in VOL_GRID] for p in PRIMAS_SENS],
                    index=[f"{p:.0%}" for p in PRIMAS_SENS], columns=[f"{v:.1%}" for v in VOL_GRID])
SENS

## 18 · Call spread overlay: note hedge 25% + warrant 60% (Exhibits 1 y 4)

In [ ]:
K1, K2 = S*(1+PRIMA_CONV), S*(1+PRIMA_WARRANT)
N_ACC = MONTO/K1; VOL_W = SIGMA + AJUSTE_VOL_WARR
COSTO_HEDGE  = float(bsm(S, K1, RF, SIGMA, PLAZO_CONV, Q_USO)["call"])*N_ACC
INGRESO_WARR = float(bsm(S, K2, RF, VOL_W, PLAZO_CONV, Q_USO)["call"])*N_ACC
NETO = COSTO_HEDGE - INGRESO_WARR
C60 = valuar_convertible(S, PRIMA_WARRANT, RF, SIGMA, PLAZO_CONV, YTM, Q_USO)
VP_EXTRA = (valor_bono(C60["cupon"], YTM, PLAZO_CONV, 1000, 2, False) - valor_bono(C["cupon"], YTM, PLAZO_CONV, 1000, 2, False))*MONTO/1000
EXH4 = pd.DataFrame({"% del principal": [COSTO_HEDGE/MONTO, INGRESO_WARR/MONTO, NETO/MONTO],
                     "USD M": [COSTO_HEDGE/1e6, INGRESO_WARR/1e6, NETO/1e6], "Caso Molson %": [0.186, 0.099, 0.086]},
                    index=["Costo note hedge (compra calls K=25%)", "Ingreso warrants (venta calls K=60%)", "Costo neto call spread"])
print(f"Cupón directo al 60%: {C60['cupon']:.3%} · VP cupón extra USD {VP_EXTRA/1e6:,.1f} M vs costo neto spread USD {NETO/1e6:,.1f} M")
EXH4

## 19 · Señal de volatilidad y straddle ATM a ~6 meses (estrategia de la clase)
σ GARCH > implícita → opción barata → comprar volatilidad (straddle largo) · σ GARCH < implícita → vender con cautela.

In [ ]:
K_ATM, T_ATM = round(S), HORIZON_DAYS/252
RATIO_VOL = VOL_GARCH_6M/IV
if RATIO_VOL > 1 + UMBRAL_SENAL:   SENAL = "COMPRAR volatilidad: straddle largo (GARCH ve más volatilidad que el mercado)"
elif RATIO_VOL < 1 - UMBRAL_SENAL: SENAL = "VENDER volatilidad con cautela (GARCH ve menos volatilidad que el mercado)"
else:                              SENAL = "SIN VENTAJA clara (GARCH y mercado coinciden)"

OPC_ATM = pd.DataFrame([{"Volatilidad": nom, "σ": s_,
                         "Call americana": griegas_crr(S, K_ATM, R_6M, s_, T_ATM, N_ARBOL, "call", True, Q_USO)["precio"],
                         "Put americana": griegas_crr(S, K_ATM, R_6M, s_, T_ATM, N_ARBOL, "put", True, Q_USO)["precio"],
                         "Put europea": crr(S, K_ATM, R_6M, s_, T_ATM, N_ARBOL, "put", False, Q_USO)[0]}
                        for nom, s_ in [("Histórica 1 año", VOL_HIST_1A), ("GARCH promedio 6m", VOL_GARCH_6M), ("Implícita", IV)]])
OPC_ATM["Prima ejercicio anticipado put"] = OPC_ATM["Put americana"] - OPC_ATM["Put europea"]

costo_str = crr(S, K_ATM, R_6M, IV, T_ATM, N_ARBOL, "call", False, Q_USO)[0] + crr(S, K_ATM, R_6M, IV, T_ATM, N_ARBOL, "put", False, Q_USO)[0]
ST_s = simular(S, HORIZON_DAYS, N_SIM*2, MU_ANUAL, np.random.default_rng(SEED + 2))[-1]
PNL = np.abs(ST_s - K_ATM) - costo_str
STRADDLE = dict(costo=costo_str, pnl_esp=PNL.mean(), prob_gan=(PNL > 0).mean(), be_inf=K_ATM - costo_str, be_sup=K_ATM + costo_str)
print(f"σ GARCH 6m {VOL_GARCH_6M:.1%} | σ implícita {IV:.1%} | razón {RATIO_VOL:.2f} → {SENAL}")
print(f"Straddle K={K_ATM}: costo {costo_str:.2f}, P&L esperado {PNL.mean():.2f}, prob. ganar {(PNL>0).mean():.1%}")
OPC_ATM

## 19B · Tarea Clase 6 — Opciones americanas de 1–3 meses sobre AAPL (portafolio Investopedia)
**Objetivo:** con los pronósticos de GARCH y Monte Carlo elegir estrategias de opciones **americanas** con vencimiento de 1–3 meses,
valuarlas con **árbol binomial CRR** (mostrando niveles de nodos), proyectar la **rentabilidad al 13-nov-2026** (2 semanas desde la
ejecución del 4-nov) y generar la **lista de órdenes** para Investopedia (PDF en la sección 22B).

| Estrategia | Vencimiento | Lógica con el forecast |
|---|---|---|
| E1 · Bull call spread | ~1.5 meses | μ > 0 en el Monte Carlo GARCH → sesgo alcista con riesgo acotado; el short call se ubica a +1σ GARCH |
| E2 · Bull put spread (crédito) | ~2.5 meses | σ implícita ≥ σ GARCH → la prima está "cara": vendemos volatilidad con riesgo acotado; short put a −0.75σ |
| E3 · Covered call | ~1.5 meses | Comprar acciones y vender call a +1σ: cobra la prima rica y captura el alza hasta el strike |

Capital de referencia: **USD 900,000**. Todos los riesgos son definidos (sin ventas descubiertas).

In [ ]:
# ===== Tarea 6 · parámetros, vencimientos y cotizaciones =====
CAPITAL_INVESTOPEDIA = 900_000
FECHA_EJEC = pd.Timestamp("2026-11-04")      # día de ejecución en Investopedia
FECHA_EVAL = pd.Timestamp("2026-11-13")      # fecha límite de evaluación (2 semanas)
PRESUPUESTO = {"E1": 0.03, "E2": 0.03, "E3": 0.08}   # E1/E2: pérdida máxima como % del capital · E3: capital en acciones
DURACION_ORDEN = "Good 'Til Cancelled (GTC)"
ANCHO_PUT = 20                                # ancho del bull put spread (USD)
N_SIM_T6 = 10000

def tercer_viernes(y, m):
    d = pd.Timestamp(y, m, 15)
    return d + pd.Timedelta(days=(4 - d.weekday()) % 7)

def mensuales(desde, n=6):
    out, y, m = [], desde.year, desde.month
    for _ in range(n):
        out.append(tercer_viernes(y, m))
        m += 1
        if m > 12:
            y, m = y + 1, 1
    return out

EXPS_LISTADAS, TK = [], None
try:
    import yfinance as yf
    TK = yf.Ticker("AAPL")
    EXPS_LISTADAS = [pd.Timestamp(e) for e in TK.options]
except Exception as e:
    print("Yahoo sin cadena de opciones → se usan precios de modelo:", str(e)[:80])

def elegir_venc(dmin, dmax, objetivo):
    cands = [e for e in mensuales(FECHA_EJEC) if dmin <= (e - FECHA_EJEC).days <= dmax]
    if EXPS_LISTADAS:
        listados = [e for e in cands if e in EXPS_LISTADAS]
        cands = listados or [e for e in EXPS_LISTADAS if dmin <= (e - FECHA_EJEC).days <= dmax] or cands
    return min(cands, key=lambda e: abs((e - FECHA_EJEC).days - objetivo))

EXP_1 = elegir_venc(30, 60, 45)
EXP_2 = elegir_venc(60, 95, 75)

v, f_ = first_value("US3MT=RR", ["MID_YLD_1", "B_YLD_1", "TR.MidYield"])
R_CORTO = registra("r_3m", en_pct(v), f"Refinitiv US3MT=RR {f_}") if v is not None else registra("r_3m", UST[2], "UST 2a (aprox.)")

CADENAS = {}
def cadena(exp):
    if exp not in CADENAS:
        res = (None, None)
        if TK is not None:
            try:
                ch = TK.option_chain(exp.strftime("%Y-%m-%d"))
                res = (ch.calls, ch.puts)
            except Exception as e:
                print("  cadena", exp.date(), str(e)[:80])
        CADENAS[exp] = res
    return CADENAS[exp]

def strike_listado(exp, tipo, K):
    calls, puts = cadena(exp)
    df = calls if tipo == "call" else puts
    if df is None or df.empty:
        return float(5*round(K/5))
    return float(df.strike.iloc[(df.strike - K).abs().argsort().iloc[0]])

def tick(p):
    return round(round(p/0.01)*0.01, 2) if p < 3 else round(round(p/0.05)*0.05, 2)

def dias_habiles(a, b):
    return int(np.busday_count(a.date(), b.date()))

def cotizar(exp, tipo, K):
    T = max((exp - FECHA_EJEC).days, 1)/365
    calls, puts = cadena(exp)
    df = calls if tipo == "call" else puts
    q = dict(bid=np.nan, ask=np.nan, last=np.nan, iv=np.nan, oi=np.nan)
    if df is not None and not df.empty and (df.strike == K).any():
        r_ = df[df.strike == K].iloc[0]
        q.update(bid=r_.get("bid", np.nan), ask=r_.get("ask", np.nan), last=r_.get("lastPrice", np.nan),
                 iv=r_.get("impliedVolatility", np.nan), oi=r_.get("openInterest", np.nan))
    iv_ok = float(q["iv"]) if pd.notna(q["iv"]) and 0.05 < q["iv"] < 1.5 else IV
    modelo = crr(S, K, R_CORTO, iv_ok, T, 300, tipo, True, Q)[0]
    if pd.notna(q["bid"]) and pd.notna(q["ask"]) and q["bid"] > 0 and q["ask"] > 0:
        mid, fuente = (q["bid"] + q["ask"])/2, "Yahoo bid/ask"
    elif pd.notna(q["last"]) and q["last"] > 0:
        mid, fuente = q["last"], "Yahoo último precio"
    else:
        mid, fuente = modelo, "Modelo CRR (σ implícita)"
    vg = vol_promedio(dias_habiles(FECHA_EJEC, exp))
    return dict(exp=exp, tipo=tipo, K=K, T=T, iv=iv_ok, mid=float(mid), limite=tick(float(mid)), fuente=fuente,
                bid=q["bid"], ask=q["ask"], oi=q["oi"], modelo_iv=float(modelo), sigma_garch=vg,
                modelo_garch=float(crr(S, K, R_CORTO, vg, T, 300, tipo, True, Q)[0]))

T1, T2 = (EXP_1 - FECHA_EJEC).days/365, (EXP_2 - FECHA_EJEC).days/365
VG1, VG2 = vol_promedio(dias_habiles(FECHA_EJEC, EXP_1)), vol_promedio(dias_habiles(FECHA_EJEC, EXP_2))
MOV1, MOV2 = S*VG1*np.sqrt(T1), S*VG2*np.sqrt(T2)
K_E1_L = strike_listado(EXP_1, "call", S)
K_E1_S = strike_listado(EXP_1, "call", S + MOV1)
K_E2_S = strike_listado(EXP_2, "put", S - 0.75*MOV2)
K_E2_L = strike_listado(EXP_2, "put", K_E2_S - ANCHO_PUT)

PATAS = {
    "E1": [("Buy to Open", +1, cotizar(EXP_1, "call", K_E1_L)), ("Sell to Open", -1, cotizar(EXP_1, "call", K_E1_S))],
    "E2": [("Buy to Open", +1, cotizar(EXP_2, "put", K_E2_L)), ("Sell to Open", -1, cotizar(EXP_2, "put", K_E2_S))],
    "E3": [("Sell to Open", -1, cotizar(EXP_1, "call", K_E1_S))],
}
print(f"AAPL ref. {S:.2f} · r 3m {R_CORTO:.2%} · σ implícita {IV:.1%}")
print(f"Vencimiento 1: {EXP_1:%d-%b-%Y} ({(EXP_1-FECHA_EJEC).days} días, σ GARCH {VG1:.1%}) · "
      f"Vencimiento 2: {EXP_2:%d-%b-%Y} ({(EXP_2-FECHA_EJEC).days} días, σ GARCH {VG2:.1%})")
pd.DataFrame([{"Estrategia": e, "Acción": a, "Tipo": c["tipo"], "Vencimiento": c["exp"].date(), "Strike": c["K"],
               "Bid": c["bid"], "Ask": c["ask"], "Mid": c["mid"], "Límite": c["limite"], "Fuente": c["fuente"],
               "σ implícita": c["iv"], "σ GARCH": c["sigma_garch"], "CRR σ impl.": c["modelo_iv"], "CRR σ GARCH": c["modelo_garch"]}
              for e, ps in PATAS.items() for a, s_, c in ps])

### Árbol binomial CRR: niveles de nodos (Hull cap. 13)
Árbol de **6 pasos** solo para visualizar (el precio usa 500 pasos). En cada nodo: precio de AAPL (S) y valor de la opción (V).
Nodos en rojo = **ejercer anticipadamente es óptimo** (relevante en la put americana).

In [ ]:
def arbol_nodos(S0, K, r, sig, T, N, tipo, q=0.0):
    dt_ = T/N
    u_ = np.exp(sig*np.sqrt(dt_)); d_ = 1/u_
    p = (np.exp((r - q)*dt_) - d_)/(u_ - d_); disc = np.exp(-r*dt_)
    sg = 1 if tipo == "call" else -1
    Sm = [[S0*u_**(i - j)*d_**j for j in range(i + 1)] for i in range(N + 1)]
    V = [None]*(N + 1); E = [None]*(N + 1)
    V[N] = [max(sg*(s - K), 0.0) for s in Sm[N]]
    E[N] = [False for v in V[N]]   # al vencimiento no es ejercicio anticipado
    for i in range(N - 1, -1, -1):
        V[i], E[i] = [], []
        for j in range(i + 1):
            cont = disc*(p*V[i + 1][j] + (1 - p)*V[i + 1][j + 1])
            ex = max(sg*(Sm[i][j] - K), 0.0)
            V[i].append(max(cont, ex)); E[i].append(i < N and ex > cont + 1e-9)
    filas = [dict(Paso=i, Día=round(i*dt_*365), Nodo=j, S=Sm[i][j], V=V[i][j], Ejercer=E[i][j])
             for i in range(N + 1) for j in range(i + 1)]
    return pd.DataFrame(filas), dict(u=u_, d=d_, p=p, dt_dias=dt_*365)

ARBOLES = {}
for nom, (acc, s_, c) in [("E1 · call largo", PATAS["E1"][0]), ("E2 · put corto", PATAS["E2"][1])]:
    df_, par_ = arbol_nodos(S, c["K"], R_CORTO, c["iv"], c["T"], 6, c["tipo"], Q)
    ARBOLES[nom] = dict(df=df_, par=par_, c=c,
                        precio500=crr(S, c["K"], R_CORTO, c["iv"], c["T"], 500, c["tipo"], True, Q)[0],
                        europea=crr(S, c["K"], R_CORTO, c["iv"], c["T"], 500, c["tipo"], False, Q)[0])
    print(f"{nom}: AAPL {c['exp']:%d-%b-%Y} {c['K']:.0f} {c['tipo']} · u={par_['u']:.4f} d={par_['d']:.4f} p={par_['p']:.4f} "
          f"Δt={par_['dt_dias']:.1f} días · árbol 6 pasos {df_.V.iloc[0]:.2f} · árbol 500 pasos {ARBOLES[nom]['precio500']:.2f} "
          f"· prima ejercicio anticipado {ARBOLES[nom]['precio500'] - ARBOLES[nom]['europea']:.3f}")

FIGS_T6 = {}
for nom, a in ARBOLES.items():
    df_ = a["df"]
    f = go.Figure()
    for _, r_ in df_[df_.Paso < df_.Paso.max()].iterrows():
        for dj in (0, 1):
            h = df_[(df_.Paso == r_.Paso + 1) & (df_.Nodo == r_.Nodo + dj)].iloc[0]
            f.add_trace(go.Scatter(x=[r_.Día, h.Día], y=[r_.S, h.S], mode="lines", line=dict(color="#cbd5e1", width=1), showlegend=False, hoverinfo="skip"))
    f.add_trace(go.Scatter(x=df_.Día, y=df_.S, mode="markers+text", showlegend=False,
                           marker=dict(size=13, color=["#c62828" if e else "#1f4e79" for e in df_.Ejercer]),
                           text=[f"{s_:.0f}<br>{v_:.2f}" for s_, v_ in zip(df_.S, df_.V)], textposition="top center", textfont=dict(size=10)))
    c = a["c"]
    f.update_layout(title=f"Árbol CRR {nom}: AAPL {c['exp']:%d-%b-%Y} {c['K']:.0f} {c['tipo'].title()} (S arriba, V abajo)",
                    xaxis_title="Días desde el 4-nov", yaxis_title="Precio AAPL", template="plotly_white", height=460,
                    margin=dict(l=50, r=20, t=50, b=40))
    FIGS_T6[f"arbol_{nom[:2]}"] = f
FIGS_T6["arbol_E1"].show()
ARBOLES["E1 · call largo"]["df"].pivot(index="Nodo", columns="Paso", values="V").round(2)

### Tamaño, riesgo y P&L proyectado al 13-nov (Monte Carlo GARCH + revaluación con árbol CRR)
Se simulan trayectorias de AAPL con GARCH(1,1) (deriva real μ) hasta el 13-nov y cada pata se **revalúa con el árbol americano**.
Dos escenarios de volatilidad al cierre: **IV constante** y **IV converge a GARCH**.

In [ ]:
def unidad(e):
    ps = PATAS[e]
    if e == "E1":
        L, Sh = ps[0][2], ps[1][2]
        neto = L["limite"] - Sh["limite"]
        return dict(neto=-neto*100, perdida=neto*100, ganancia=(Sh["K"] - L["K"] - neto)*100, be=L["K"] + neto)
    if e == "E2":
        L, Sh = ps[0][2], ps[1][2]
        cred = Sh["limite"] - L["limite"]
        return dict(neto=cred*100, perdida=(Sh["K"] - L["K"] - cred)*100, ganancia=cred*100, be=Sh["K"] - cred)
    c = ps[0][2]
    return dict(neto=-(S - c["limite"])*100, perdida=(S - c["limite"])*100, ganancia=(c["K"] - S + c["limite"])*100, be=S - c["limite"])

UNID = {e: unidad(e) for e in PATAS}
CANT = {}
for e in PATAS:
    base = UNID[e]["perdida"] if e != "E3" else S*100
    CANT[e] = max(1, int(PRESUPUESTO[e]*CAPITAL_INVESTOPEDIA // max(base, 1)))

H_T6 = dias_habiles(FECHA_EJEC, FECHA_EVAL)
ST6 = simular(S, H_T6, N_SIM_T6, MU_ANUAL, np.random.default_rng(SEED + 6))[-1]
SG = np.linspace(np.percentile(ST6, 0.1)*0.98, np.percentile(ST6, 99.9)*1.02, 81)

def valor_eval(c, escenario):
    T_rem = max((c["exp"] - FECHA_EVAL).days, 1)/365
    sig = c["iv"] if escenario == "IV constante" else vol_promedio(dias_habiles(FECHA_EVAL, c["exp"]))
    vals = np.array([crr(s_, c["K"], R_CORTO, sig, T_rem, 120, c["tipo"], True, Q)[0] for s_ in SG])
    return np.interp(ST6, SG, vals)

PNL = {}
for esc in ("IV constante", "IV → GARCH"):
    for e, ps in PATAS.items():
        pnl = np.zeros_like(ST6)
        for acc, sgn, c in ps:
            pnl += sgn*(valor_eval(c, esc) - c["limite"])*100
        if e == "E3":
            pnl += (ST6 - S)*100
        PNL[(e, esc)] = pnl*CANT[e]

def greeks_estrategia(e):
    tot = dict(Delta=0.0, Gamma=0.0, Theta=0.0, Vega=0.0)
    for acc, sgn, c in PATAS[e]:
        g_ = griegas_crr(S, c["K"], R_CORTO, c["iv"], c["T"], 300, c["tipo"], True, Q)
        for k_, kk in (("Delta", "delta"), ("Gamma", "gamma"), ("Theta", "theta"), ("Vega", "vega")):
            tot[k_] += sgn*g_[kk]*100*CANT[e]
    if e == "E3":
        tot["Delta"] += 100*CANT[e]
    return tot

NOMBRES = {"E1": "Bull call spread", "E2": "Bull put spread (crédito)", "E3": "Covered call"}
filas = []
for e in PATAS:
    u_, n_ = UNID[e], CANT[e]
    pb, pg = PNL[(e, "IV constante")], PNL[(e, "IV → GARCH")]
    riesgo = u_["perdida"]*n_
    filas.append({"Estrategia": f"{e} · {NOMBRES[e]}", "Unidades": n_,
                  "Flujo inicial USD": u_["neto"]*n_, "Pérdida máx. USD": riesgo, "Ganancia máx. USD": u_["ganancia"]*n_,
                  "Break-even": u_["be"], "E[P&L] 13-nov (IV const.)": pb.mean(), "E[P&L] 13-nov (IV→GARCH)": pg.mean(),
                  "Prob. ganancia": (pb > 0).mean(), "P5 USD": np.percentile(pb, 5), "P95 USD": np.percentile(pb, 95),
                  "Rend. esperado s/riesgo": pb.mean()/riesgo if riesgo else np.nan,
                  **greeks_estrategia(e),
                  "Decisión": "EJECUTAR" if pb.mean() > 0 or pg.mean() > 0 else "NO EJECUTAR"})
RES_T6 = pd.DataFrame(filas)
tot_b = sum(PNL[(e, "IV constante")] for e in PATAS if RES_T6.set_index("Estrategia").loc[f"{e} · {NOMBRES[e]}", "Decisión"] == "EJECUTAR")
PORT_T6 = dict(e_pnl=float(np.mean(tot_b)) if np.ndim(tot_b) else 0.0, prob=float(np.mean(np.asarray(tot_b) > 0)) if np.ndim(tot_b) else 0.0,
               p5=float(np.percentile(tot_b, 5)) if np.ndim(tot_b) else 0.0, p95=float(np.percentile(tot_b, 95)) if np.ndim(tot_b) else 0.0)
PORT_T6["rend_capital"] = PORT_T6["e_pnl"]/CAPITAL_INVESTOPEDIA
print(f"Días hábiles 4-nov → 13-nov: {H_T6} · AAPL 13-nov (MC GARCH): mediana {np.median(ST6):.2f}, P5 {np.percentile(ST6,5):.2f}, P95 {np.percentile(ST6,95):.2f}")
print(f"Portafolio de opciones: E[P&L] al 13-nov USD {PORT_T6['e_pnl']:,.0f} ({PORT_T6['rend_capital']:.2%} del capital) · "
      f"prob. ganancia {PORT_T6['prob']:.0%} · P5 {PORT_T6['p5']:,.0f} · P95 {PORT_T6['p95']:,.0f}")
RES_T6.T

In [ ]:
# ===== Lista de órdenes para Investopedia y plan de salida =====
def instrumento(c):
    return f"AAPL {c['exp']:%d-%b-%Y} {c['K']:.0f} {'Call' if c['tipo'] == 'call' else 'Put'}"

ORD, n_ord = [], 0
for e in PATAS:
    if RES_T6.loc[RES_T6.Estrategia.str.startswith(e), "Decisión"].iloc[0] != "EJECUTAR":
        continue
    n_ = CANT[e]
    if e == "E3":
        n_ord += 1
        ORD.append({"#": n_ord, "Estrategia": f"{e} · {NOMBRES[e]}", "Acción Investopedia": "Buy (acciones)", "Instrumento": "AAPL acciones",
                    "Cantidad": 100*n_, "Tipo de orden": "Limit", "Precio límite": round(S, 2), "Duración": DURACION_ORDEN,
                    "Monto USD": -S*100*n_, "Nota": "Primero las acciones: así el call vendido queda cubierto"})
    pasos = sorted(PATAS[e], key=lambda x: 0 if x[0] == "Buy to Open" else 1)
    for acc, sgn, c in pasos:
        n_ord += 1
        ORD.append({"#": n_ord, "Estrategia": f"{e} · {NOMBRES[e]}", "Acción Investopedia": acc, "Instrumento": instrumento(c),
                    "Cantidad": n_, "Tipo de orden": "Limit", "Precio límite": c["limite"], "Duración": DURACION_ORDEN,
                    "Monto USD": (-1 if sgn > 0 else 1)*c["limite"]*100*n_,
                    "Nota": ("Pata larga primero (cubre la corta)" if sgn > 0 and e != "E3" else "Pata corta; colócala tras llenar la larga" if e != "E3" else "Vender call cubierto")})
ORDENES = pd.DataFrame(ORD)

SALIDA = []
for e in PATAS:
    u_ = UNID[e]
    if e == "E1":
        SALIDA.append({"Estrategia": f"{e} · {NOMBRES[e]}",
                       "Toma de utilidad": f"Cerrar si el spread vale ≥ {(-u_['neto'] + 0.5*u_['ganancia'])/100:.2f} (50% de la ganancia máx.)",
                       "Stop": f"Cerrar si el spread vale ≤ {(-u_['neto'])/200:.2f} (pierde 50% del débito)",
                       "Cierre": "13-nov: Sell to Close el call largo y Buy to Close el call corto"})
    elif e == "E2":
        SALIDA.append({"Estrategia": f"{e} · {NOMBRES[e]}",
                       "Toma de utilidad": f"Cerrar si recomprar el spread cuesta ≤ {u_['neto']/200:.2f} (50% del crédito)",
                       "Stop": f"Cerrar si recomprar cuesta ≥ {2*u_['neto']/100:.2f} (2× el crédito) o AAPL < {PATAS['E2'][1][2]['K']:.0f}",
                       "Cierre": "13-nov: Buy to Close la put corta y Sell to Close la put larga"})
    else:
        SALIDA.append({"Estrategia": f"{e} · {NOMBRES[e]}",
                       "Toma de utilidad": f"Si AAPL ≥ {PATAS['E3'][0][2]['K']:.0f}: cerrar ambas patas (ganancia máxima casi alcanzada)",
                       "Stop": f"Si AAPL < {S*(1 - 2*VG1*np.sqrt(H_T6/252)):.2f} (−2σ GARCH a 2 semanas): vender acciones y Buy to Close el call",
                       "Cierre": "13-nov: Buy to Close el call y Sell las acciones (o mantener si se decide continuar)"})
SALIDA = pd.DataFrame(SALIDA)

FIGS_T6["pnl"] = go.Figure()
for e, col in zip(PATAS, ("#2e75b6", "#2e7d32", "#b7791f")):
    FIGS_T6["pnl"].add_trace(go.Histogram(x=PNL[(e, "IV constante")], name=NOMBRES[e], opacity=0.55, nbinsx=60, marker_color=col))
FIGS_T6["pnl"].add_vline(x=0, line_color="#6b7280")
FIGS_T6["pnl"].update_layout(barmode="overlay", title="Distribución del P&L al 13-nov (Monte Carlo GARCH, IV constante)",
                             xaxis_title="USD", template="plotly_white", height=420, legend=dict(orientation="h", y=-0.2))

def _f(k): return FIGS_T6[k].to_html(full_html=False, include_plotlyjs=False, config={"displayModeBar": False})
def _t(df): return df.to_html(border=0, float_format=lambda x: f"{x:,.2f}", na_rep="n/d", index=False)
patas_df = pd.DataFrame([{"Estrategia": e, "Acción": a, "Instrumento": instrumento(c), "Bid": c["bid"], "Ask": c["ask"], "Límite": c["limite"],
                          "Fuente": c["fuente"], "σ impl.": c["iv"], "σ GARCH": c["sigma_garch"], "CRR σ impl.": c["modelo_iv"],
                          "CRR σ GARCH": c["modelo_garch"]} for e, ps in PATAS.items() for a, s_, c in ps])
T6_HTML = (
    f'<p><b>Ejecución:</b> {FECHA_EJEC:%d-%b-%Y} · <b>Evaluación:</b> {FECHA_EVAL:%d-%b-%Y} ({H_T6} días hábiles) · '
    f'<b>Capital:</b> USD {CAPITAL_INVESTOPEDIA:,.0f} · AAPL ref. {S:.2f} · σ impl. {IV:.1%} · σ GARCH 1–3m {VG1:.1%}–{VG2:.1%}</p>'
    f'<p><b>Portafolio de opciones:</b> E[P&L] al 13-nov USD {PORT_T6["e_pnl"]:,.0f} ({PORT_T6["rend_capital"]:.2%} del capital), '
    f'probabilidad de ganancia {PORT_T6["prob"]:.0%}, P5 USD {PORT_T6["p5"]:,.0f} · P95 USD {PORT_T6["p95"]:,.0f}.</p>'
    f'<h3>Órdenes para Investopedia</h3>{_t(ORDENES)}<h3>Resultados por estrategia</h3>{_t(RES_T6)}'
    f'<h3>Plan de salida</h3>{_t(SALIDA)}<h3>Cotizaciones y valuación por pata</h3>{_t(patas_df)}'
    f'<div class="two"><div class="fig">{_f("arbol_E1")}</div><div class="fig">{_f("arbol_E2")}</div></div><div class="fig">{_f("pnl")}</div>'
)
ORDENES

## 20 · Gráficas

In [ ]:
PAL = dict(azul="#1f4e79", azul2="#2e75b6", verde="#2e7d32", rojo="#c62828", gris="#6b7280", oro="#b7791f")
LAY = dict(template="plotly_white", font=dict(family="Inter, Segoe UI, Arial", size=12), margin=dict(l=50, r=20, t=50, b=40),
           height=380, legend=dict(orientation="h", y=-0.22))
FIGS = {}
pctax = dict(tickformat=".0%")

f = go.Figure(go.Scatter(x=HIST.index, y=HIST.values, name="AAPL", line=dict(color=PAL["azul"])))
for val, nom, col in [(S, "Spot", PAL["gris"]), (K1, f"Conversión {PRIMA_CONV:.0%}", PAL["verde"]), (K2, f"Warrant {PRIMA_WARRANT:.0%}", PAL["rojo"])]:
    f.add_hline(y=val, line_dash="dash", line_color=col, annotation_text=f"{nom} ${val:,.0f}")
f.update_layout(title="AAPL — precio vs precios de conversión", **LAY); FIGS["precio"] = f

f = go.Figure(go.Scatter(x=list(ust_x), y=np.array(ust_y)*100, name="UST", mode="lines+markers", line=dict(color=PAL["gris"])))
UU = U.dropna(subset=["ytm"])
f.add_trace(go.Scatter(x=UU["anios"], y=UU["ytm"]*100, mode="markers", name="Bonos AAPL", text=UU["nombre"], marker=dict(color=PAL["azul2"], size=9)))
f.add_trace(go.Scatter(x=[T_BONO], y=[YTM*100], mode="markers", name="Bono elegido", marker=dict(color=PAL["rojo"], size=15, symbol="star")))
f.update_layout(title="Curva USD de Apple vs Treasuries", xaxis_title="Años", yaxis_title="Rendimiento (%)", **LAY); FIGS["curva"] = f

idx = REND.index
f = go.Figure()
f.add_trace(go.Scatter(x=idx, y=VOL_30.values, name="Histórica 30 días", line=dict(color=PAL["gris"], width=1)))
f.add_trace(go.Scatter(x=idx, y=vol_anual(V_EW), name="EWMA λ=0.94", line=dict(color="black", width=1, dash="dash")))
f.add_trace(go.Scatter(x=idx, y=vol_anual(V_G), name="GARCH(1,1)", line=dict(color=PAL["azul2"], width=2)))
f.add_hline(y=float(vol_anual(VL)), line_dash="dot", line_color=PAL["rojo"], annotation_text=f"V_L {vol_anual(VL):.1%}")
f.update_layout(title="AAPL: volatilidad anualizada por modelo", yaxis=pctax, **LAY); FIGS["vol_modelos"] = f

dd = np.arange(1, int(PLAZO_CONV*252) + 1)
f = go.Figure(go.Scatter(x=dd, y=vol_anual(pron_var(len(dd))), name="σ diaria pronosticada", line=dict(color=PAL["azul2"], width=2)))
f.add_trace(go.Scatter(x=dd, y=[vol_promedio(d) for d in dd], name="σ promedio hasta el día", line=dict(color=PAL["oro"], width=2, dash="dash")))
f.add_hline(y=float(vol_anual(VL)), line_dash="dot", line_color=PAL["rojo"], annotation_text="Largo plazo")
f.add_vline(x=HORIZON_DAYS, line_color=PAL["gris"], annotation_text="6 meses"); f.add_vline(x=int(PLAZO_CONV*252), line_color=PAL["verde"], annotation_text="6 años")
f.update_layout(title="Reversión a la media: estructura temporal GARCH", xaxis_title="Días hábiles", yaxis=pctax, **LAY); FIGS["vol_term"] = f

f = go.Figure()
if not CORR_EWMA.empty:
    f.add_trace(go.Scatter(x=CORR_60.index, y=CORR_60.values, name="Móvil 60 días", line=dict(color=PAL["gris"])))
    f.add_trace(go.Scatter(x=CORR_EWMA.index, y=CORR_EWMA.values, name="EWMA λ=0.94", line=dict(color="black", width=2)))
f.update_layout(title="Correlación AAPL – S&P 500", **LAY); FIGS["corr"] = f

dias = np.arange(HORIZON_DAYS + 1); pc = np.percentile(SIM, [5, 25, 50, 75, 95], axis=1)
f = go.Figure()
f.add_trace(go.Scatter(x=dias, y=pc[4], line=dict(width=0), showlegend=False))
f.add_trace(go.Scatter(x=dias, y=pc[0], fill="tonexty", fillcolor="rgba(46,117,182,.15)", line=dict(width=0), name="P5–P95"))
f.add_trace(go.Scatter(x=dias, y=pc[3], line=dict(width=0), showlegend=False))
f.add_trace(go.Scatter(x=dias, y=pc[1], fill="tonexty", fillcolor="rgba(46,117,182,.35)", line=dict(width=0), name="P25–P75"))
f.add_trace(go.Scatter(x=dias, y=pc[2], name="Mediana", line=dict(color=PAL["azul"], width=2)))
f.add_hline(y=S, line_dash="dot", line_color=PAL["gris"]); f.add_hline(y=K1, line_dash="dash", line_color=PAL["verde"], annotation_text="Conversión")
f.update_layout(title=f"Abanico Monte Carlo GARCH a {HORIZON_DAYS} días", xaxis_title="Días hábiles", **LAY); FIGS["abanico"] = f

ib = REND.index[INICIO_BT:]; lim = -Z_BT*np.sqrt(VG_BT[INICIO_BT:]); exc = u[INICIO_BT:] < lim
f = go.Figure(go.Scatter(x=ib, y=u[INICIO_BT:], name="Rendimiento diario (%)", line=dict(color=PAL["gris"], width=1)))
f.add_trace(go.Scatter(x=ib, y=lim, name="−VaR 99% GARCH", line=dict(color=PAL["azul2"], width=1.6)))
f.add_trace(go.Scatter(x=ib[exc], y=u[INICIO_BT:][exc], mode="markers", name="Excepciones", marker=dict(color="black", size=7)))
f.update_layout(title="Back testing del VaR 99% a 1 día", **LAY); FIGS["backtest"] = f

f = go.Figure(go.Bar(x=VOL_COMP["Volatilidad"], y=VOL_COMP["opcion_bono"], marker_color=[PAL["rojo"] if e else PAL["azul2"] for e in VOL_COMP["En uso"]],
                     text=[f"σ {s_:.1%}<br>cupón {c_:.2%}" for s_, c_ in zip(VOL_COMP["σ"], VOL_COMP["cupon"])], textposition="outside"))
f.update_layout(title="Valor de la opción de conversión por bono según la volatilidad", yaxis_title="USD por bono", **LAY); FIGS["opc_vol"] = f

f = go.Figure(go.Bar(x=EXH3["Año"], y=EXH3["Flujo"], name="Flujo", marker_color=PAL["azul2"]))
f.add_trace(go.Bar(x=EXH3["Año"], y=EXH3["VP"], name="Valor presente", marker_color=PAL["oro"]))
f.update_layout(title=f"Bono puro: cupón {C['cupon']:.2%} descontado al {YTM:.2%} → ${C['piso']:.1f}", barmode="group", yaxis_type="log", xaxis_title="Año", **LAY); FIGS["flujos"] = f

Sx = np.linspace(0.3*S, 2.5*S, 200)
f = go.Figure(go.Scatter(x=Sx, y=np.full_like(Sx, C["piso"]), name="Piso (bono puro)", line=dict(color=PAL["gris"], dash="dash")))
f.add_trace(go.Scatter(x=Sx, y=C["n"]*Sx, name="Paridad de conversión", line=dict(color=PAL["verde"], dash="dot")))
f.add_trace(go.Scatter(x=Sx, y=C["piso"] + C["n"]*bsm(Sx, C["K"], RF, SIGMA, PLAZO_CONV, Q_USO)["call"], name="Convertible", line=dict(color=PAL["azul"], width=3)))
f.add_vline(x=S, line_color=PAL["rojo"], annotation_text="Spot")
f.update_layout(title="Convertible = bono puro + calls (σ GARCH)", xaxis_title="Precio AAPL", yaxis_title="USD por bono", **LAY); FIGS["convertible"] = f

f = make_subplots(specs=[[{"secondary_y": True}]])
f.add_trace(go.Bar(x=[f"{p:.0%}" for p in SENS["Prima"]], y=SENS["opcion_bono"], name="Opción por bono", marker_color=PAL["azul2"]), secondary_y=False)
f.add_trace(go.Scatter(x=[f"{p:.0%}" for p in SENS["Prima"]], y=SENS["cupon"]*100, name="Cupón a la par (%)", line=dict(color=PAL["rojo"], width=3)), secondary_y=True)
f.update_layout(title="Menor prima → opción más valiosa → cupón más bajo", **LAY); FIGS["prima"] = f

f = go.Figure(go.Heatmap(z=HEAT.values, x=HEAT.columns, y=HEAT.index, colorscale="RdYlGn_r", text=np.round(HEAT.values, 2), texttemplate="%{text}%", colorbar=dict(title="Cupón %")))
f.update_layout(title="Cupón a la par: prima × volatilidad", xaxis_title="Volatilidad", yaxis_title="Prima", **{**LAY, "height": 420}); FIGS["heat"] = f

STg = np.linspace(0.5*S, 2.5*S, 200); pay = np.maximum(STg-K1, 0) - np.maximum(STg-K2, 0)
f = go.Figure(go.Scatter(x=STg, y=pay, name="Payoff call spread", line=dict(color=PAL["verde"], width=3)))
f.add_trace(go.Scatter(x=STg, y=pay - NETO/N_ACC*np.exp(RF*PLAZO_CONV), name="Neto de costo (VF)", line=dict(color=PAL["azul"], dash="dash")))
f.add_trace(go.Scatter(x=STg, y=np.maximum(STg-K1, 0), name="Dilución sin overlay", line=dict(color=PAL["rojo"], dash="dot")))
f.update_layout(title="Overlay: compra call 25% + venta warrant 60%", xaxis_title="AAPL al vencimiento", yaxis_title="USD por acción", **LAY); FIGS["spread"] = f

gx = bsm(Sx, C["K"], RF, SIGMA, PLAZO_CONV, Q_USO)
f = make_subplots(specs=[[{"secondary_y": True}]])
f.add_trace(go.Scatter(x=Sx, y=gx["delta_c"], name="Delta", line=dict(color=PAL["azul"])), secondary_y=False)
f.add_trace(go.Scatter(x=Sx, y=gx["vega"]*C["n"], name="Vega por bono", line=dict(color=PAL["verde"], dash="dot")), secondary_y=True)
f.update_layout(title="Griegas de la opción de conversión", xaxis_title="Precio AAPL", **LAY); FIGS["griegas"] = f

gr = np.linspace(np.percentile(ST_s, 0.5), np.percentile(ST_s, 99.5), 200)
f = make_subplots(specs=[[{"secondary_y": True}]])
f.add_trace(go.Histogram(x=ST_s, nbinsx=60, name="Distribución GARCH", marker_color="rgba(107,114,128,.35)", histnorm="probability density"), secondary_y=True)
f.add_trace(go.Scatter(x=gr, y=np.abs(gr - K_ATM) - costo_str, name="P&L straddle", line=dict(color=PAL["azul"], width=3)), secondary_y=False)
f.add_hline(y=0, line_color=PAL["gris"])
f.update_layout(title=f"Straddle ATM K={K_ATM} y distribución GARCH a 6 meses", xaxis_title="Precio al vencimiento", **LAY); FIGS["straddle"] = f
FIGS["vol_modelos"].show()

## 21 · Dashboard HTML (FINA 405 + caso Molson + GARCH)

In [ ]:
def fm(v, tipo="num", d=2):
    if v is None or (isinstance(v, float) and np.isnan(v)): return "n/d"
    return {"pct": f"{v*100:,.{d}f}%", "usd": f"${v:,.{d}f}", "bn": f"${v/1e9:,.1f} B", "x": f"{v:,.{d}f}x"}.get(tipo, f"{v:,.{d}f}")
def card(t, v, sub="", alerta=False):
    return f'<div class="card{" warn" if alerta else ""}"><div class="lab">{t}</div><div class="val">{v}</div><div class="sub">{sub}</div></div>'
def fuente(k): return FUENTES.get(k, "")
def esresp(k): return "respaldo" in FUENTES.get(k, "") or "SIMULADOS" in FUENTES.get(k, "")
def tabla(df): return df.to_html(classes="tbl", float_format=lambda x: f"{x:,.4f}", border=0, na_rep="n/d")
def fig(k): return f'<div class="fig">{FIGS[k].to_html(full_html=False, include_plotlyjs=False, config={"displayModeBar": False})}</div>'

ahorro_pb = (YTM - C["cupon"])*1e4
bt99 = BT[(BT["Confianza"] == "99%") & (BT["Modelo"] == "GARCH(1,1)")].iloc[0]

kpi_resumen = "".join([
    card("Bono base", "4.500% 2032", BONO["isin"]),
    card("Precio limpio", fm(precio_bono, d=3), fuente("precio_bono"), esresp("precio_bono")),
    card("YTM (tasa descuento)", fm(YTM, "pct"), "Análogo al 5.75% del caso"),
    card("Spread vs UST", f"{SPREAD*1e4:,.0f} pb", f"Dur. mod. {MOD:.2f}"),
    card("AAPL spot", fm(S, "usd"), fuente("S"), esresp("S")),
    card("σ convertible", fm(SIGMA, "pct", 1), fuente("vol")),
    card("rf 6 años", fm(RF, "pct"), fuente("rf"), esresp("ust5")),
    card("Dividend yield", fm(Q, "pct"), fuente("q"), esresp("q")),
    card("Opción conversión/bono", fm(C["opcion_bono"], "usd", 1), f"{C['n']:.3f} acc. × ${C['call_acc']:.2f}"),
    card("Bono puro", fm(C["piso"], "usd", 1), "Piso del convertible"),
    card("Cupón a la par", fm(C["cupon"], "pct"), f"Ahorro {ahorro_pb:,.0f} pb vs straight"),
    card("Call spread neto", fm(NETO/MONTO, "pct", 1), f"USD {NETO/1e6:,.1f} M · caso 8.6%"),
])
kpi_vol = "".join([
    card("α (shock)", f"{ALPHA:.4f}", "GARCH(1,1) MLE"), card("β (memoria)", f"{BETA:.4f}", f"Persistencia {ALPHA+BETA:.3f}"),
    card("Vida media shock", f"{VIDA_MEDIA:.0f} días", "ln 0.5 / ln(α+β)"),
    card("σ largo plazo", fm(float(vol_anual(VL)), "pct", 1), "V_L anualizada"),
    card("σ GARCH hoy", fm(float(vol_anual(V_NEXT)), "pct", 1), "Día siguiente"),
    card("σ histórica 1 año", fm(VOL_HIST_1A, "pct", 1), fuente("historia"), esresp("historia")),
    card("σ GARCH promedio 6 años", fm(VOL_GARCH_6A, "pct", 1), "Entra al convertible" if VOL_FUENTE == "GARCH" else "Referencia"),
    card("σ implícita ATM", fm(IV, "pct", 1), fuente("iv"), esresp("iv")),
])
kpi_riesgo = "".join([
    card("Mediana 6 meses", fm(MC["Mediana"], "usd"), f"P5 {MC['P5']:,.0f} · P95 {MC['P95']:,.0f}"),
    card("Prob. de subir", fm(MC["Prob. sube"], "pct", 1), f"μ supuesto {MU_ANUAL:.0%}"),
    card("VaR 99% 10 días", fm(MC["VaR 99% 10 días (%)"], "pct", 1), "Monte Carlo GARCH"),
    card("Kupiec VaR 99% GARCH", f"p = {bt99['p Kupiec']:.3f}", f"{bt99['Excepciones']} excepciones · {bt99['Resultado']}", bt99["p Kupiec"] < 0.05),
    card("Corr. EWMA vs S&P", fm(CORR_EWMA.iloc[-1] if not CORR_EWMA.empty else np.nan), f"Beta 1 año {BETA_MKT:.2f}"),
    card("Mejor modelo (AIC)", MEJOR_MODELO, "Hull 23.4"),
])
kpi_emp = "".join([card(k + (" (estimada)" if k == "Interest Coverage" and COB_EST else ""), fm(EMP[k], t),
                        "EBIT / (deuda × YTM)" if k == "Interest Coverage" and COB_EST else "Refinitiv") for k, t in [
    ("Market Cap", "bn"), ("Revenue (LTM)", "bn"), ("EBITDA", "bn"), ("Net Income", "bn"), ("Total Debt", "bn"), ("Cash & ST Inv.", "bn"),
    ("D/E", "x"), ("Debt/Assets", "pct"), ("Interest Coverage", "x"), ("Net Debt/EBITDA", "x"), ("EBIT Margin", "pct"), ("Net Margin", "pct")]])
kpi_pais = "".join([card("USD/CNY", fm(PAIS["USD/CNY"], d=4), "Spot"), card("China 10Y", fm(PAIS["China 10Y"], "pct"), "Gov. yield"),
                    card("US 10Y", fm(PAIS["US 10Y"], "pct"), "Treasury"), card("Diferencial CN−US", fm(PAIS["China 10Y"] - PAIS["US 10Y"], "pct"), "Puntos de tasa")])

concl = f"""
<p><b>Bono base.</b> {BONO['nombre']} ({BONO['isin']}): precio {precio_bono:.3f}, YTM {YTM:.2%}, {SPREAD*1e4:.0f} pb sobre UST,
duración modificada {MOD:.2f}. Es el análogo del bono straight a 6 años de Molson (5.75% en el caso).</p>
<p><b>Volatilidad (clase GARCH).</b> GARCH(1,1) estimado por máxima verosimilitud sobre {len(u):,} rendimientos diarios: α = {ALPHA:.3f},
β = {BETA:.3f}, persistencia {ALPHA+BETA:.3f} (un shock se diluye a la mitad en ~{VIDA_MEDIA:.0f} días). Hoy la volatilidad es {vol_anual(V_NEXT):.1%} y revierte hacia
{vol_anual(VL):.1%}; por eso la σ promedio para la vida del convertible (6 años) es <b>{VOL_GARCH_6A:.1%}</b>, frente a {VOL_HIST_1A:.1%} histórica de 1 año
y {IV:.1%} implícita. El mejor modelo por AIC es <i>{MEJOR_MODELO}</i>. Usar la σ de hoy subestimaría la opción: GARCH justifica una σ de largo plazo.</p>
<p><b>Convertible sintético a {PLAZO_CONV:.0f} años, prima {PRIMA_CONV:.0%}.</b> Con σ = {SIGMA:.1%} la opción vale ${C['opcion_bono']:.1f} por bono
({C['n']:.3f} acciones a ${C['K']:.2f}); el bono puro debe valer ${C['piso']:.1f}, lo que exige un cupón de <b>{C['cupon']:.2%}</b> vs {YTM:.2%} straight:
ahorro de {ahorro_pb:.0f} pb (≈ USD {(YTM-C['cupon'])*MONTO/1e6:,.1f} M al año sobre USD {MONTO/1e9:.1f} B). El árbol CRR americano agrega solo
${(am_['precio']-eu_)*C['n']:.2f} por bono de prima de ejercicio anticipado (q = {Q:.2%} ≪ r = {RF:.2%}), y el Monte Carlo GARCH (Duan) da
${MC_CALL*C['n']:.1f}: Black-Scholes es una aproximación válida, igual que en el caso.</p>
<p><b>Riesgo.</b> Para el inversionista, el VaR 99% a 10 días de la acción es {MC['VaR 99% 10 días (%)']:.1%}; el back testing del VaR 99% GARCH
{'pasa' if bt99['p Kupiec'] > 0.05 else 'no pasa'} la prueba de Kupiec (p = {bt99['p Kupiec']:.3f}). La dilución potencial es de solo
{fm(EMP.get('Dilución conv. 25%'), 'pct')} de las acciones: para Apple el convertible es una herramienta de costo, no de capital.</p>
<p><b>Call spread.</b> Note hedge {COSTO_HEDGE/MONTO:.1%} − warrants {INGRESO_WARR/MONTO:.1%} = costo neto <b>{NETO/MONTO:.1%}</b> del principal (caso: 8.6%).
Emitir directo al 60% exigiría cupón {C60['cupon']:.2%}; el VP de ese cupón extra (USD {VP_EXTRA/1e6:,.1f} M) frente al costo del spread (USD {NETO/1e6:,.1f} M)
confirma el punto del caso: <i>no hay free lunch</i>.</p>
<p><b>Estrategia de volatilidad.</b> σ GARCH 6m {VOL_GARCH_6M:.1%} vs implícita {IV:.1%} (razón {RATIO_VOL:.2f}): {SENAL}.
Straddle ATM K={K_ATM}: costo {STRADDLE['costo']:.2f}, P&L esperado {STRADDLE['pnl_esp']:.2f}, probabilidad de ganar {STRADDLE['prob_gan']:.0%}.</p>
<p><b>Limitaciones.</b> GARCH(1,1) supone normalidad y no capta asimetría (GJR/EGARCH); μ es un supuesto; el bono real tiene make-whole call
(Molson era non-call life) y Apple no tiene convertibles: es una emisión hipotética.</p>"""

tabs = [
 ("Resumen", f'<div class="grid">{kpi_resumen}</div><div class="two">{fig("convertible")}{fig("precio")}</div>'),
 ("Bonos AAPL", f'<p class="nota">Fuente: {FUENTE_UNIV}. Puntaje = cercanía a 6 años y a la par.</p>{fig("curva")}'
     + tabla(U.sort_values("puntaje", ascending=False)[["nombre", "cusip", "isin", "cupon", "vencimiento", "anios", "precio", "ytm", "puntaje"]].reset_index(drop=True))),
 ("Empresa & País", f'<h3>Apple Inc. — perfil financiero</h3><div class="grid">{kpi_emp}</div><h3>Contexto China</h3><div class="grid">{kpi_pais}</div>'),
 ("Volatilidad GARCH", f'<div class="grid">{kpi_vol}</div><div class="two">{fig("vol_modelos")}{fig("vol_term")}</div>'
     f'<h3>Parámetros y validación</h3><p class="nota">{ARCH_CHECK}</p>{tabla(GARCH_TAB.to_frame("AAPL"))}'
     f'<h3>Elección de modelo (Hull 23.4)</h3>{tabla(COMPAR.set_index("Modelo"))}<h3>Estructura temporal</h3>{tabla(EST_TEMP)}'),
 ("Riesgo & VaR", f'<div class="grid">{kpi_riesgo}</div><div class="two">{fig("abanico")}{fig("backtest")}</div>{fig("corr")}'
     f'<h3>Back testing (Kupiec)</h3>{tabla(BT.set_index(["Confianza", "Modelo"]))}<h3>Monte Carlo GARCH</h3>{tabla(MC.to_frame("AAPL"))}'),
 ("Opción de conversión", f'{fig("opc_vol")}{tabla(VOL_COMP.set_index("Volatilidad"))}'
     f'<h3>Exhibit 2 — Black-Scholes-Merton (q = {Q_USO:.2%})</h3>{tabla(EXH2)}'
     f'<h3>Americana (árbol CRR, N={N_ARBOL}) y Monte Carlo GARCH</h3>{tabla(AMER)}{tabla(GRIEGAS_CRR.to_frame("Griegas CRR americana (por acción)"))}{fig("griegas")}'),
 ("Bono puro & cupón", f'<h3>Exhibit 3</h3>{fig("flujos")}{tabla(EXH3)}<p class="nota">Convertible = ${C["piso"]:.2f} + ${C["opcion_bono"]:.2f} = ${C["piso"]+C["opcion_bono"]:.2f}</p>'),
 ("Sensibilidades", f'<div class="two">{fig("prima")}{fig("heat")}</div>{tabla(SENS)}'),
 ("Call spread", f'<h3>Exhibits 1 y 4</h3>{tabla(EXH4)}{fig("spread")}<p class="nota">Cupón directo al {PRIMA_WARRANT:.0%}: {C60["cupon"]:.3%} · σ warrant {VOL_W:.1%}</p>'),
 ("Estrategia volatilidad", f'<p><b>Señal:</b> {SENAL} (σ GARCH 6m {VOL_GARCH_6M:.1%} / σ implícita {IV:.1%} = {RATIO_VOL:.2f})</p>'
     f'{fig("straddle")}{tabla(OPC_ATM.set_index("Volatilidad"))}<p class="nota">Puntos de equilibrio {STRADDLE["be_inf"]:.2f} y {STRADDLE["be_sup"]:.2f}</p>'),
 ("Validación Molson", f'<p class="nota">El modelo reproduce los exhibits del caso con sus mismos supuestos.</p>{tabla(VALID.set_index("Concepto"))}'),
 ("Conclusiones", concl),
]
tabs.insert(len(tabs) - 2, ("Tarea 6 · Opciones 1–3m", T6_HTML))
nav = "".join(f'<button class="tab{" on" if i == 0 else ""}" onclick="ver({i})">{t}</button>' for i, (t, _) in enumerate(tabs))
body = "".join(f'<section id="s{i}" style="display:{"block" if i == 0 else "none"}">{c}</section>' for i, (_, c) in enumerate(tabs))
fuentes_txt = " · ".join(f"{k}: {v}" for k, v in FUENTES.items() if not k.startswith("emp_"))
aviso = '<div class="aviso">⚠ Vista generada con datos simulados/respaldo — no son datos de mercado. Corre el notebook con tu sesión de Refinitiv.</div>' if esresp("historia") else ""

HTML = f"""<!doctype html><html lang="es"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>AAPL Convertible GARCH</title><script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
:root{{--bg:#f5f7fa;--card:#fff;--ink:#1a202c;--mut:#6b7280;--acc:#1f4e79;--warn:#b45309}}
body{{margin:0;background:var(--bg);color:var(--ink);font-family:Inter,Segoe UI,Arial,sans-serif}}
header{{background:linear-gradient(135deg,#0f2742,#1f4e79);color:#fff;padding:22px 28px}}
header h1{{margin:0;font-size:22px}} header p{{margin:4px 0 0;opacity:.85;font-size:13px}}
.aviso{{background:#fef3c7;color:#92400e;padding:10px 28px;font-size:13px;font-weight:600}}
nav{{display:flex;flex-wrap:wrap;gap:6px;padding:12px 28px;background:#fff;border-bottom:1px solid #e5e7eb;position:sticky;top:0;z-index:5}}
.tab{{border:1px solid #d1d5db;background:#fff;border-radius:18px;padding:6px 14px;cursor:pointer;font-size:13px;color:var(--ink)}}
.tab.on{{background:var(--acc);color:#fff;border-color:var(--acc)}}
main{{padding:20px 28px;max-width:1400px;margin:auto}}
.grid{{display:grid;grid-template-columns:repeat(auto-fill,minmax(200px,1fr));gap:12px;margin-bottom:18px}}
.card{{background:var(--card);border-radius:10px;padding:14px 16px;box-shadow:0 1px 3px rgba(0,0,0,.08);border-left:4px solid var(--acc)}}
.card.warn{{border-left-color:var(--warn)}} .lab{{font-size:12px;color:var(--mut);text-transform:uppercase;letter-spacing:.04em}}
.val{{font-size:22px;font-weight:700;margin:4px 0}} .sub{{font-size:11px;color:var(--mut)}}
.two{{display:grid;grid-template-columns:repeat(auto-fit,minmax(min(480px,100%),1fr));gap:14px}}
.fig{{background:#fff;border-radius:10px;padding:6px;margin-bottom:14px}}
table{{border-collapse:collapse;background:#fff;font-size:12.5px;margin:10px 0 18px;width:100%;overflow-x:auto;display:block}}
th,td{{padding:6px 10px;border-bottom:1px solid #eef0f3;text-align:right;white-space:nowrap}} th{{background:#f0f4f8;color:#334155}}
h3{{margin:18px 0 6px;color:#0f2742}} .nota{{font-size:12.5px;color:var(--mut)}} section p{{line-height:1.55;max-width:1050px}}
footer{{font-size:11px;color:var(--mut);padding:16px 28px}}
@media(max-width:600px){{main,nav,header,.aviso{{padding-left:16px;padding-right:16px}}}}
</style></head><body>
<header><h1>Apple Inc. (AAPL) — Bono Convertible Sintético con volatilidad GARCH</h1>
<p>{CURSO} · {AUTORES} · Base: caso Molson Coors (NE90-06-107) + Hull caps. 13, 22 y 23 · Datos: Refinitiv/LSEG · {HOY:%d-%b-%Y}</p></header>
{aviso}<nav>{nav}</nav><main>{body}</main>
<footer>Fuentes: {fuentes_txt}. Las tarjetas con borde ámbar usan valores de respaldo. Material académico; no es recomendación de inversión.</footer>
<script>function ver(i){{document.querySelectorAll("section").forEach((s,j)=>s.style.display=j==i?"block":"none");
document.querySelectorAll(".tab").forEach((b,j)=>b.classList.toggle("on",j==i));window.dispatchEvent(new Event("resize"));}}</script>
</body></html>"""

RUTA = os.path.join(CARPETA, "AAPL_Convertible_GARCH_Dashboard.html")
with open(RUTA, "w", encoding="utf-8") as fh: fh.write(HTML)
print("Dashboard guardado en:", os.path.abspath(RUTA))
from IPython.display import IFrame, display
display(IFrame(src=RUTA, width="100%", height=900))

## 22 · Exportar tablas a Excel

In [ ]:
RUTA_XLSX = os.path.join(CARPETA, "AAPL_Convertible_GARCH_Tablas.xlsx")
with pd.ExcelWriter(RUTA_XLSX, engine="openpyxl") as xw:
    for nombre, df_, idx_ in [("Universo_bonos", U, False), ("GARCH_parametros", GARCH_TAB.to_frame("AAPL"), True),
                              ("Eleccion_modelo", COMPAR, False), ("Estructura_vol", EST_TEMP, True), ("MonteCarlo", MC.to_frame("AAPL"), True),
                              ("Backtest_VaR", BT, False), ("Vol_vs_opcion", VOL_COMP, False), ("Exhibit2_Opcion", EXH2, True),
                              ("Americana_CRR_MC", AMER, True), ("Exhibit3_Bono", EXH3, False), ("Sensibilidad_prima", SENS, False),
                              ("Heatmap_cupon", HEAT, True), ("Exhibit4_CallSpread", EXH4, True), ("Opcion_ATM_6m", OPC_ATM, False),
                              ("Validacion_Molson", VALID, False), ("T6_Ordenes", ORDENES, False), ("T6_Resultados", RES_T6, False), ("T6_Salida", SALIDA, False),
                              ("T6_Arbol_E1", ARBOLES["E1 · call largo"]["df"], False), ("T6_Arbol_E2", ARBOLES["E2 · put corto"]["df"], False), ("Fuentes", pd.Series(FUENTES).to_frame("Fuente"), True)]:
        df_.to_excel(xw, sheet_name=nombre, index=idx_)
print("Excel exportado en:", os.path.abspath(RUTA_XLSX))

## 22B · PDF de órdenes para Investopedia (ejecución 4-nov-2026)

In [ ]:
try:
    import reportlab
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "reportlab"])
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from reportlab.lib.pagesizes import letter, landscape
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak

def png_arbol(nom, ruta):
    a = ARBOLES[nom]; df_ = a["df"]; c = a["c"]
    fig_, ax = plt.subplots(figsize=(9, 4.6))
    for _, r_ in df_[df_.Paso < df_.Paso.max()].iterrows():
        for dj in (0, 1):
            h = df_[(df_.Paso == r_.Paso + 1) & (df_.Nodo == r_.Nodo + dj)].iloc[0]
            ax.plot([r_.Día, h.Día], [r_.S, h.S], color="#cbd5e1", lw=1, zorder=1)
    ax.scatter(df_.Día, df_.S, c=["#c62828" if e else "#1f4e79" for e in df_.Ejercer], s=60, zorder=2)
    for _, r_ in df_.iterrows():
        ax.annotate(f"{r_.S:.0f}\n{r_.V:.2f}", (r_.Día, r_.S), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=7)
    ax.set_title(f"Árbol CRR {nom}: AAPL {c['exp']:%d-%b-%Y} {c['K']:.0f} {c['tipo'].title()} (S / V; rojo = ejercer)", fontsize=10)
    ax.set_xlabel("Días desde el 4-nov"); ax.set_ylabel("AAPL"); fig_.tight_layout(); fig_.savefig(ruta, dpi=150); plt.close(fig_)

def png_pnl(ruta):
    fig_, ax = plt.subplots(figsize=(9, 3.8))
    for e, col in zip(PATAS, ("#2e75b6", "#2e7d32", "#b7791f")):
        ax.hist(PNL[(e, "IV constante")], bins=60, alpha=0.5, color=col, label=NOMBRES[e])
    ax.axvline(0, color="gray"); ax.legend(fontsize=8); ax.set_title("P&L al 13-nov por estrategia (Monte Carlo GARCH)", fontsize=10)
    ax.set_xlabel("USD"); fig_.tight_layout(); fig_.savefig(ruta, dpi=150); plt.close(fig_)

os.makedirs(os.path.join(CARPETA, "pdf_tmp"), exist_ok=True)
IMG = {k: os.path.join(CARPETA, "pdf_tmp", f"{k}.png") for k in ("E1", "E2", "pnl")}
png_arbol("E1 · call largo", IMG["E1"]); png_arbol("E2 · put corto", IMG["E2"]); png_pnl(IMG["pnl"])

ss = getSampleStyleSheet()
H1 = ParagraphStyle("h1", parent=ss["Title"], fontSize=17, textColor=colors.HexColor("#0f2742"))
H2 = ParagraphStyle("h2", parent=ss["Heading2"], textColor=colors.HexColor("#1f4e79"))
TX = ParagraphStyle("tx", parent=ss["BodyText"], fontSize=9, leading=12)
CEL = ParagraphStyle("cel", parent=ss["BodyText"], fontSize=7.5, leading=9)
AV = ParagraphStyle("av", parent=TX, textColor=colors.HexColor("#92400e"), backColor=colors.HexColor("#fef3c7"))

def tabla_pdf(df, anchos=None, fmt="{:,.2f}"):
    data = [[Paragraph(f"<b>{c}</b>", CEL) for c in df.columns]]
    for _, r_ in df.iterrows():
        data.append([Paragraph(fmt.format(v) if isinstance(v, (float, np.floating)) else str(v), CEL) for v in r_.values])
    t = Table(data, colWidths=anchos, repeatRows=1)
    t.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#e8eef5")), ("GRID", (0, 0), (-1, -1), 0.3, colors.HexColor("#cbd5e1")),
                           ("VALIGN", (0, 0), (-1, -1), "TOP"), ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#f8fafc")])]))
    return t

RUTA_PDF = os.path.join(CARPETA, f"Ordenes_Investopedia_AAPL_{FECHA_EJEC:%d%b%Y}.pdf")
doc = SimpleDocTemplate(RUTA_PDF, pagesize=landscape(letter), leftMargin=0.5*inch, rightMargin=0.5*inch, topMargin=0.5*inch, bottomMargin=0.5*inch)
el = []
el.append(Paragraph(f"Órdenes para Investopedia · AAPL · ejecución {FECHA_EJEC:%A %d-%b-%Y}", H1))
el.append(Paragraph(f"{CURSO} · {AUTORES} · Tarea Clase 6 (GARCH + Monte Carlo + árboles binomiales) · Generado {pd.Timestamp.now():%d-%b-%Y %H:%M}", TX))
if API is None or esresp("historia"):
    el.append(Paragraph("⚠ GENERADO SIN CONEXIÓN A LSEG: precios de respaldo. NO usar para operar; vuelve a correr el notebook con la sesión abierta.", AV))
el.append(Spacer(1, 8))
el.append(Paragraph("Resumen del mercado y del modelo", H2))
resumen = pd.DataFrame([
    ["AAPL referencia", f"{S:.2f}", fuente("S")], ["σ implícita", f"{IV:.1%}", fuente("iv")],
    ["σ GARCH hoy / largo plazo", f"{float(vol_anual(V_NEXT)):.1%} / {float(vol_anual(VL)):.1%}", f"α={ALPHA:.3f}, β={BETA:.3f}, persistencia {ALPHA+BETA:.3f}"],
    [f"σ GARCH promedio a {EXP_1:%d-%b} / {EXP_2:%d-%b}", f"{VG1:.1%} / {VG2:.1%}", "Hull 23.6 (volatilidad promedio para la vida de la opción)"],
    ["Tasa libre de riesgo corta", f"{R_CORTO:.2%}", fuente("r_3m")], ["Dividend yield", f"{Q:.2%}", fuente("q")],
    ["AAPL al 13-nov (MC GARCH)", f"mediana {np.median(ST6):.2f}", f"P5 {np.percentile(ST6,5):.2f} · P95 {np.percentile(ST6,95):.2f}"],
    ["Capital Investopedia", f"USD {CAPITAL_INVESTOPEDIA:,.0f}", "Presupuesto de riesgo: E1 3%, E2 3%, E3 8% en acciones"],
], columns=["Concepto", "Valor", "Fuente / nota"])
el.append(tabla_pdf(resumen, [2.6*inch, 2.0*inch, 5.4*inch]))
el.append(Spacer(1, 8))
el.append(Paragraph(f"<b>Horario:</b> el mercado abre a las 9:30 ET = <b>8:30 hora de Monterrey</b>. Las opciones solo se operan en horario regular. "
                    f"Las órdenes son <b>Limit · {DURACION_ORDEN}</b>: si no se llenan quedan en espera. "
                    f"<b>Regla de control:</b> antes de enviar cada orden compara el precio límite con la cotización (bid/ask) de Investopedia; "
                    f"si el mid difiere más de 5% o AAPL abre más de 2% lejos de {S:.2f}, vuelve a correr el notebook y usa los nuevos límites.", TX))
el.append(PageBreak())

el.append(Paragraph("1 · Órdenes a colocar (en este orden)", H2))
ORD_PDF = ORDENES.copy(); ORD_PDF["Monto USD"] = ORD_PDF["Monto USD"].map(lambda x: f"{x:,.0f}")
el.append(tabla_pdf(ORD_PDF, [0.3*inch, 1.4*inch, 1.0*inch, 1.7*inch, 0.75*inch, 0.6*inch, 0.65*inch, 1.2*inch, 0.85*inch, 1.55*inch]))
el.append(Spacer(1, 6))
neto = ORDENES["Monto USD"].sum() if len(ORDENES) else 0
el.append(Paragraph(f"Flujo neto de efectivo al abrir todo: <b>USD {neto:,.0f}</b> (negativo = sale efectivo). "
                    f"Monto (−) = pagas prima / compras; (+) = recibes prima.", TX))
el.append(Spacer(1, 6))
el.append(Paragraph("Cómo capturarlas en Investopedia", H2))
for paso in ["Trade → <b>Options</b> → símbolo <b>AAPL</b> → elige <b>Expiration</b>, <b>Call/Put</b> y <b>Strike</b> exactamente como en la columna Instrumento.",
             "<b>Action</b>: Buy to Open / Sell to Open según la tabla · <b>Quantity</b>: número de contratos (1 contrato = 100 acciones).",
             "<b>Order Type</b>: Limit · <b>Limit Price</b>: el de la tabla · <b>Duration</b>: Good 'Til Cancelled.",
             "En spreads: primero la pata larga (Buy to Open) y, ya llenada, la corta (Sell to Open). En el covered call: primero las acciones.",
             "Preview Order → revisa el costo/crédito → Submit. Toma screenshot de la confirmación y de 'Open Trades'."]:
    el.append(Paragraph("• " + paso, TX))
el.append(PageBreak())

el.append(Paragraph("2 · Detalle por estrategia (riesgo, forecast y griegas)", H2))
RES_PDF = RES_T6[["Estrategia", "Unidades", "Flujo inicial USD", "Pérdida máx. USD", "Ganancia máx. USD", "Break-even",
                  "E[P&L] 13-nov (IV const.)", "E[P&L] 13-nov (IV→GARCH)", "Prob. ganancia", "Delta", "Theta", "Vega", "Decisión"]].copy()
RES_PDF["Prob. ganancia"] = RES_PDF["Prob. ganancia"].map(lambda x: f"{x:.0%}")
el.append(tabla_pdf(RES_PDF))
el.append(Spacer(1, 6))
justif = {
    "E1": f"Sesgo alcista moderado: el Monte Carlo GARCH (μ={MU_ANUAL:.0%}) da mediana {np.median(ST6):.2f} al 13-nov. El short call en {K_E1_S:.0f} (+1σ GARCH) abarata la entrada y reduce la exposición a vega.",
    "E2": f"σ implícita {IV:.1%} vs σ GARCH {VG2:.1%}: la prima se vende 'cara'. Strike corto {K_E2_S:.0f} ≈ −0.75σ; la put larga {K_E2_L:.0f} limita la pérdida. Gana con el paso del tiempo (theta).",
    "E3": f"Compra de {100*CANT['E3']} acciones con venta de call {K_E1_S:.0f}: cobra prima y participa del alza hasta el strike; el delta neto se reduce frente a tener solo acciones.",
}
for e in PATAS:
    el.append(Paragraph(f"<b>{e} · {NOMBRES[e]}:</b> {justif[e]}", TX))
el.append(Spacer(1, 6))
el.append(Paragraph(f"<b>Portafolio:</b> E[P&L] al 13-nov USD {PORT_T6['e_pnl']:,.0f} ({PORT_T6['rend_capital']:.2%} del capital), probabilidad de ganancia "
                    f"{PORT_T6['prob']:.0%}, P5 USD {PORT_T6['p5']:,.0f} y P95 USD {PORT_T6['p95']:,.0f}. Es un pronóstico del modelo, no una garantía.", TX))
el.append(Image(IMG["pnl"], width=8.5*inch, height=3.6*inch))
el.append(PageBreak())

el.append(Paragraph("3 · Plan de salida (evaluación al 13-nov-2026)", H2))
el.append(tabla_pdf(SALIDA, [1.6*inch, 2.9*inch, 2.9*inch, 2.6*inch]))
el.append(Spacer(1, 8))
el.append(Paragraph("4 · Árboles binomiales CRR (niveles de nodos)", H2))
for nom, k in (("E1 · call largo", "E1"), ("E2 · put corto", "E2")):
    a = ARBOLES[nom]
    el.append(Paragraph(f"<b>{nom}</b>: u={a['par']['u']:.4f}, d={a['par']['d']:.4f}, p={a['par']['p']:.4f}, Δt={a['par']['dt_dias']:.1f} días. "
                        f"Árbol 6 pasos {a['df'].V.iloc[0]:.2f} · 500 pasos {a['precio500']:.2f} · europea {a['europea']:.2f} · "
                        f"prima de ejercicio anticipado {a['precio500'] - a['europea']:.3f}.", TX))
    el.append(Image(IMG[k], width=8.6*inch, height=4.4*inch))
el.append(PageBreak())

el.append(Paragraph("5 · Cotizaciones y valuación por pata", H2))
pat_pdf = pd.DataFrame([{"Estrategia": e, "Acción": a, "Instrumento": instrumento(c), "Bid": c["bid"], "Ask": c["ask"], "Límite": c["limite"],
                         "Fuente": c["fuente"], "σ impl.": f"{c['iv']:.1%}", "σ GARCH": f"{c['sigma_garch']:.1%}",
                         "CRR σ impl.": c["modelo_iv"], "CRR σ GARCH": c["modelo_garch"]} for e, ps in PATAS.items() for a, s_, c in ps])
el.append(tabla_pdf(pat_pdf))
el.append(Spacer(1, 8))
el.append(Paragraph("6 · Evidencias (screenshots a tomar)", H2))
for s_ in ["<b>Investopedia:</b> cada 'Preview/Confirm' de orden, 'Open Trades' (pendientes) y 'Holdings' después del llenado; Portfolio Overview el 4-nov y el 13-nov.",
           f"<b>CME Group:</b> página de opciones/volatilidad del Nasdaq-100 (QuikStrike o CVOL) como contexto de volatilidad del mercado.",
           f"<b>TradingView:</b> gráfico diario de AAPL con los strikes {K_E1_L:.0f}, {K_E1_S:.0f}, {K_E2_S:.0f} y {K_E2_L:.0f} marcados y los break-even.",
           "<b>Notebook:</b> salida de la celda de conexión (CONECTADO), tabla de órdenes y árbol de nodos."]:
    el.append(Paragraph("• " + s_, TX))
el.append(Spacer(1, 10))
el.append(Paragraph("Material académico con dinero simulado (Investopedia). Los resultados del modelo son pronósticos con supuestos (GARCH normal, μ supuesto, IV constante o convergente) y no garantizan ganancias.", TX))
doc.build(el)

from pathlib import Path
DESC = Path.home()/"Downloads"; DESC.mkdir(exist_ok=True)
shutil_ok = False
try:
    import shutil; shutil.copy(RUTA_PDF, DESC/Path(RUTA_PDF).name); shutil_ok = True
except Exception as e:
    print("No se copió a Descargas:", e)
print("PDF generado:", os.path.abspath(RUTA_PDF), "| copia en Descargas:" if shutil_ok else "", DESC/Path(RUTA_PDF).name if shutil_ok else "")

## 23 · Descargar el dashboard y el notebook completo en HTML
Guarda el notebook (**Cmd+S**) antes de correr esta celda para que el HTML del código incluya las últimas salidas.

In [ ]:
import glob, base64, shutil
from pathlib import Path
from IPython.display import HTML as _HTML

DESCARGAS = Path.home()/"Downloads"; DESCARGAS.mkdir(exist_ok=True)
dash = Path(RUTA).resolve(); dash_dest = DESCARGAS/dash.name; shutil.copy(dash, dash_dest)

nb = next((Path(p).resolve() for p in sorted(glob.glob("*.ipynb"), key=os.path.getmtime, reverse=True)
           if "VOL_GARCH_6A" in open(p, encoding="utf-8").read()), None)
nb_html = None
if nb is None:
    print("⚠ No encontré el .ipynb en esta carpeta; solo se descarga el dashboard.")
else:
    res = subprocess.run([sys.executable, "-m", "jupyter", "nbconvert", "--to", "html", "--output", f"{nb.stem}_codigo",
                          "--output-dir", str(DESCARGAS), str(nb)], capture_output=True, text=True)
    nb_html = DESCARGAS/f"{nb.stem}_codigo.html"
    if res.returncode != 0: print(res.stderr[-800:]); nb_html = None

def boton(ruta, texto):
    b64 = base64.b64encode(Path(ruta).read_bytes()).decode()
    return (f'<a download="{Path(ruta).name}" href="data:text/html;base64,{b64}" style="display:inline-block;margin:6px 10px 6px 0;'
            f'padding:10px 18px;background:#1f4e79;color:#fff;border-radius:8px;text-decoration:none;font-weight:600">⬇ {texto}</a>')
display(_HTML(boton(dash_dest, "Descargar dashboard HTML") + (boton(nb_html, "Descargar notebook completo HTML") if nb_html else "")))
print("Copiados en Descargas:", dash_dest, nb_html or "")

In [ ]:
!jupyter nbconvert --to html AAPL_Convertible_GARCH_Proyecto.ipynb

In [ ]:
!open AAPL_Convertible_GARCH_Proyecto.html